# Notebook 6 — Next-Best-Product Recommendation

## Objective

This notebook develops a customer-level product recommendation system.

The primary business question is:

> **Given everything known about a customer at the current prediction point, which products are most likely to appear in their next qualifying merchandise order?**

This differs from the earlier predictive notebooks.

The output is not a single probability or revenue estimate.

Instead, the model must produce a ranked list of products for each customer.

For example:

> **Customer → Product 1, Product 2, Product 3, Product 4, Product 5**

The primary evaluation therefore focuses on recommendation-ranking metrics such as:

- Hit Rate@K;
- Recall@K;
- Precision@K;
- Mean Reciprocal Rank;
- and catalogue coverage.

The analysis will also distinguish between:

### Repeat / Replenishment

Products the customer has purchased previously.

### Cross-Sell / Discovery

Products that are new to that customer at the prediction point.

This distinction is important because a recommendation system may perform well simply by predicting recurring purchases without generating genuine product discovery.


## Prediction Unit and Target

The prediction unit is:

> **one customer immediately after a completed qualifying order**

At that point, only transactions up to and including that completed order may be used as recommendation history.

The outcome is:

> **the merchandise basket in that customer's next qualifying order**

The primary recommendation target therefore consists of the products that appear in the customer's next order.

The notebook will initially evaluate next-order recommendation rather than claiming to predict all products the customer will purchase over their remaining lifetime.

### Maximum Evaluation Horizon

A next order must occur within **90 days** of the prediction order to be included as an observed next-order outcome.

Prediction episodes near the end of the dataset require a complete 90-day observation horizon.

Customers without an observed next purchase within the 90-day horizon are not treated as having a known product target for ranking evaluation.

This does not mean that they permanently churned.


## Recommendation Modes

Two recommendation modes will be evaluated.

### Mode A — Next-Basket Recommendation

Products may include items the customer has purchased previously.

This reflects the broad question:

> **What is this customer likely to buy next?**

### Mode B — Cross-Sell Recommendation

Previously purchased products are excluded from the recommendation list.

This reflects the narrower question:

> **What new product is this customer likely to add next?**

The notebook will first measure the actual prevalence of repeat and new-to-customer products before deciding how heavily each mode should feature in the final system.

In [ ]:
# ================================================================
# 57.4 — IMPORTS
# ================================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ================================================================
# 57.5 — LOAD CANONICAL CLEANED DATA
# ================================================================

df = pd.read_parquet(
    "cleaned_online_retail.parquet"
)


print(
    f"Rows loaded: {len(df):,}"
)

print(
    f"Columns loaded: {df.shape[1]:,}"
)

In [ ]:
# ================================================================
# 57.6 — REQUIRED COLUMN VALIDATION
# ================================================================

required_columns = [
    "date",
    "invoice",
    "customer",
    "product",
    "description",
    "region",
    "quantity",
    "unit_price",
    "customer_is_identified",
    "is_merchandise",
    "is_cancellation",
    "net_merchandise_revenue",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)


print(
    "✓ Required recommendation columns confirmed."
)

In [ ]:
# ================================================================
# 57.7 — OBSERVATION BOUNDARY
# ================================================================

RECOMMENDATION_END_DATE = pd.Timestamp(
    "2011-11-30 23:59:59"
)


print(
    f"Recommendation observation boundary: "
    f"{RECOMMENDATION_END_DATE}"
)

In [ ]:
# ================================================================
# 57.8 — QUALIFYING POSITIVE MERCHANDISE
# ================================================================

recommendation_lines = (
    df.loc[
        df["customer_is_identified"]
        &
        df["is_merchandise"]
        &
        (df["quantity"] > 0)
        &
        (df["unit_price"] > 0)
        &
        (~df["is_cancellation"])
        &
        (df["date"] <= RECOMMENDATION_END_DATE)
    ]
    .copy()
)


recommendation_lines[
    "purchase_day"
] = (
    recommendation_lines[
        "date"
    ]
    .dt.normalize()
)


print(
    f"Qualifying merchandise rows: "
    f"{len(recommendation_lines):,}"
)

print(
    f"Customers: "
    f"{recommendation_lines['customer'].nunique():,}"
)

print(
    f"Products: "
    f"{recommendation_lines['product'].nunique():,}"
)

In [ ]:
# ================================================================
# 57.9 — CUSTOMER ORDER SUMMARY
# ================================================================

customer_orders = (
    recommendation_lines
    .groupby(
        [
            "customer",
            "invoice",
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        order_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        order_units=(
            "quantity",
            "sum"
        ),

        distinct_products=(
            "product",
            "nunique"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


customer_orders = (
    customer_orders
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Customer-order rows: "
    f"{len(customer_orders):,}"
)

print(
    f"Customers: "
    f"{customer_orders['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 57.10 — ORDER PRODUCT BASKETS
# ================================================================

order_baskets = (
    recommendation_lines
    .groupby(
        [
            "customer",
            "invoice",
        ]
    )[
        "product"
    ]
    .agg(
        lambda values:
            tuple(
                sorted(
                    set(values)
                )
            )
    )
    .rename(
        "basket_products"
    )
    .reset_index()
)


customer_orders = (
    customer_orders
    .merge(
        order_baskets,
        on=[
            "customer",
            "invoice",
        ],
        how="left",
        validate="one_to_one"
    )
)


print(
    "✓ Product baskets attached to customer orders."
)

In [ ]:
# ================================================================
# 57.11 — CUSTOMER ORDER SEQUENCE
# ================================================================

customer_orders[
    "customer_order_number"
] = (
    customer_orders
    .groupby(
        "customer"
    )
    .cumcount()
    +
    1
)


customer_orders[
    "customer_total_orders"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "invoice"
    ]
    .transform(
        "size"
    )
)


display(
    customer_orders[
        [
            "customer",
            "invoice",
            "order_date",
            "customer_order_number",
            "customer_total_orders",
            "distinct_products",
        ]
    ]
    .head()
)

In [ ]:
# ================================================================
# 57.12 — NEXT ORDER
# ================================================================

customer_orders[
    "next_invoice"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "invoice"
    ]
    .shift(-1)
)


customer_orders[
    "next_order_date"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .shift(-1)
)


customer_orders[
    "next_basket_products"
] = (
    customer_orders
    .groupby(
        "customer"
    )[
        "basket_products"
    ]
    .shift(-1)
)


customer_orders[
    "days_to_next_order"
] = (
    customer_orders[
        "next_order_date"
    ]
    -
    customer_orders[
        "order_date"
    ]
).dt.total_seconds() / 86400

In [ ]:
# ================================================================
# 57.13 — NEXT ORDER WITHIN 90 DAYS
# ================================================================

NEXT_ORDER_HORIZON_DAYS = 90


customer_orders[
    "has_next_order_within_90d"
] = (
    customer_orders[
        "next_order_date"
    ]
    .notna()
    &
    (
        customer_orders[
            "next_order_date"
        ]
        <=
        (
            customer_orders[
                "order_date"
            ]
            +
            pd.Timedelta(
                days=NEXT_ORDER_HORIZON_DAYS
            )
        )
    )
).astype(int)

In [ ]:
# ================================================================
# 57.14 — COMPLETE FOLLOW-UP
# ================================================================

customer_orders[
    "complete_90d_followup"
] = (
    customer_orders[
        "order_date"
    ]
    +
    pd.Timedelta(
        days=NEXT_ORDER_HORIZON_DAYS
    )
    <=
    RECOMMENDATION_END_DATE
)


print(
    "Complete 90-day follow-up episodes: "
    f"{customer_orders['complete_90d_followup'].sum():,}"
)

### 57.15 — Recommendation evaluation population
For product-ranking evaluation, we need an observed next basket.

Therefore the primary evaluation population is:
- complete 90-day follow-up;
- next order exists;
- next order occurred within 90 days.

Important:
This population is conditional on a next order occurring within 90 days.
Notebook 2/3 already address whether someone returns. Notebook 6 answers:
> Given that the customer purchases again, what are they likely to buy?

That keeps the modelling tasks cleanly separated.

In [ ]:
# ================================================================
# 57.15 — OBSERVED NEXT-BASKET POPULATION
# ================================================================

next_basket_episodes = (
    customer_orders.loc[
        customer_orders[
            "complete_90d_followup"
        ]
        &
        (
            customer_orders[
                "has_next_order_within_90d"
            ]
            == 1
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Observed next-basket episodes: "
    f"{len(next_basket_episodes):,}"
)

print(
    f"Unique customers: "
    f"{next_basket_episodes['customer'].nunique():,}"
)

### 57.16 — Build cumulative customer product history
We need to know what the customer had purchased before or at the prediction order.

This must be sequential and leakage-safe.

In [ ]:
# ================================================================
# 57.16 — CUSTOMER PRODUCT HISTORY BEFORE NEXT ORDER
# ================================================================

customer_product_history_records = []


for customer, group in (
    customer_orders
    .groupby(
        "customer",
        sort=False
    )
):

    purchased_products = set()


    for row in group.itertuples(
        index=False
    ):

        current_basket = set(
            row.basket_products
        )


        purchased_products.update(
            current_basket
        )


        customer_product_history_records.append(
            {
                "customer":
                    customer,

                "invoice":
                    row.invoice,

                "historical_products":
                    tuple(
                        sorted(
                            purchased_products
                        )
                    ),

                "historical_unique_product_count":
                    len(
                        purchased_products
                    ),
            }
        )

In [ ]:
customer_product_history = pd.DataFrame(
    customer_product_history_records
)


next_basket_episodes = (
    next_basket_episodes
    .merge(
        customer_product_history,
        on=[
            "customer",
            "invoice",
        ],
        how="left",
        validate="one_to_one"
    )
)


print(
    "✓ Leakage-safe customer product histories created."
)

In [ ]:
# ================================================================
# 57.17 — REPEAT VS NEW PRODUCTS IN NEXT BASKET
# ================================================================

def split_next_basket_products(
    row
):

    history = set(
        row[
            "historical_products"
        ]
    )


    next_basket = set(
        row[
            "next_basket_products"
        ]
    )


    repeat_products = (
        next_basket
        &
        history
    )


    new_products = (
        next_basket
        -
        history
    )


    return pd.Series(
        {
            "next_basket_size":
                len(
                    next_basket
                ),

            "repeat_product_count":
                len(
                    repeat_products
                ),

            "new_product_count":
                len(
                    new_products
                ),

            "repeat_products":
                tuple(
                    sorted(
                        repeat_products
                    )
                ),

            "new_products":
                tuple(
                    sorted(
                        new_products
                    )
                ),
        }
    )


next_basket_composition = (
    next_basket_episodes
    .apply(
        split_next_basket_products,
        axis=1
    )
)


next_basket_episodes = pd.concat(
    [
        next_basket_episodes,
        next_basket_composition,
    ],
    axis=1
)

In [ ]:
# ================================================================
# 57.18 — NEXT-BASKET COMPOSITION SUMMARY
# ================================================================

next_basket_episodes[
    "repeat_share"
] = (
    next_basket_episodes[
        "repeat_product_count"
    ]
    /
    next_basket_episodes[
        "next_basket_size"
    ]
)


next_basket_episodes[
    "new_product_share"
] = (
    next_basket_episodes[
        "new_product_count"
    ]
    /
    next_basket_episodes[
        "next_basket_size"
    ]
)


next_basket_composition_summary = pd.DataFrame(
    {
        "Metric": [
            "Prediction episodes",
            "Average next-basket size",
            "Median next-basket size",
            "Average repeat-product count",
            "Average new-product count",
            "Average repeat share",
            "Average new-product share",
            "Episodes containing any repeat product",
            "Episodes containing any new product",
            "All-repeat baskets",
            "All-new baskets",
        ],

        "Value": [
            len(
                next_basket_episodes
            ),

            next_basket_episodes[
                "next_basket_size"
            ].mean(),

            next_basket_episodes[
                "next_basket_size"
            ].median(),

            next_basket_episodes[
                "repeat_product_count"
            ].mean(),

            next_basket_episodes[
                "new_product_count"
            ].mean(),

            next_basket_episodes[
                "repeat_share"
            ].mean(),

            next_basket_episodes[
                "new_product_share"
            ].mean(),

            (
                next_basket_episodes[
                    "repeat_product_count"
                ]
                > 0
            ).mean(),

            (
                next_basket_episodes[
                    "new_product_count"
                ]
                > 0
            ).mean(),

            (
                next_basket_episodes[
                    "new_product_count"
                ]
                == 0
            ).mean(),

            (
                next_basket_episodes[
                    "repeat_product_count"
                ]
                == 0
            ).mean(),
        ],
    }
)


display(
    next_basket_composition_summary
)

In [ ]:
# ================================================================
# 57.19 — NEXT-BASKET SIZE DISTRIBUTION
# ================================================================

next_basket_size_summary = (
    next_basket_episodes[
        "next_basket_size"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


display(
    next_basket_size_summary
)

In [ ]:
# ================================================================
# VISUAL 57.1 — NEXT-BASKET SIZE
# ================================================================

basket_visual_cap = (
    next_basket_episodes[
        "next_basket_size"
    ]
    .quantile(
        0.99
    )
)


plt.figure(
    figsize=(10, 6)
)


sns.histplot(
    next_basket_episodes[
        "next_basket_size"
    ]
    .clip(
        upper=basket_visual_cap
    ),
    bins=40
)


plt.title(
    "Products in the Customer's Next Order"
)

plt.xlabel(
    "Distinct Products in Next Basket — capped at 99th percentile"
)

plt.ylabel(
    "Prediction Episodes"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 57.2 — REPEAT VS NEW
# ================================================================

repeat_new_visual = pd.DataFrame(
    {
        "Product Type": [
            "Previously Purchased",
            "New to Customer",
        ],

        "Average Share": [
            next_basket_episodes[
                "repeat_share"
            ].mean(),

            next_basket_episodes[
                "new_product_share"
            ].mean(),
        ],
    }
)


plt.figure(
    figsize=(8, 5)
)


sns.barplot(
    data=repeat_new_visual,
    x="Product Type",
    y="Average Share"
)


plt.title(
    "Composition of the Next Customer Basket"
)

plt.xlabel(
    ""
)

plt.ylabel(
    "Average Share of Next Basket"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 57.20 — CUSTOMER HISTORY DEPTH
# ================================================================

history_depth_summary = (
    next_basket_episodes[
        [
            "customer_order_number",
            "historical_unique_product_count",
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)


display(
    history_depth_summary
)

In [ ]:
# ================================================================
# 57.21 — DAYS TO NEXT ORDER
# ================================================================

next_order_timing_summary = (
    next_basket_episodes[
        "days_to_next_order"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)


display(
    next_order_timing_summary
)

In [ ]:
# ================================================================
# 57.22 — PRODUCT POPULARITY
# ================================================================

product_popularity = (
    recommendation_lines
    .groupby(
        "product",
        as_index=False
    )
    .agg(
        orders=(
            "invoice",
            "nunique"
        ),

        customers=(
            "customer",
            "nunique"
        ),

        revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
    )
    .sort_values(
        "orders",
        ascending=False
    )
    .reset_index(drop=True)
)


product_popularity[
    "order_share"
] = (
    product_popularity[
        "orders"
    ]
    /
    product_popularity[
        "orders"
    ].sum()
)


product_popularity[
    "cumulative_order_share"
] = (
    product_popularity[
        "order_share"
    ]
    .cumsum()
)


display(
    product_popularity.head(20)
)

In [ ]:
# ================================================================
# 57.23 — CATALOGUE CONCENTRATION
# ================================================================

catalogue_concentration_records = []


for n_products in [
    10,
    25,
    50,
    100,
    250,
    500,
]:

    top_orders = (
        product_popularity
        .head(
            n_products
        )[
            "orders"
        ]
        .sum()
    )


    catalogue_concentration_records.append(
        {
            "Top Products":
                n_products,

            "Share of Product-Order Occurrences":
                (
                    top_orders
                    /
                    product_popularity[
                        "orders"
                    ]
                    .sum()
                ),
        }
    )


catalogue_concentration = pd.DataFrame(
    catalogue_concentration_records
)


catalogue_concentration[
    "Share (%)"
] = (
    catalogue_concentration[
        "Share of Product-Order Occurrences"
    ]
    *
    100
)


display(
    catalogue_concentration
)

In [ ]:
# ================================================================
# 57.24 — PREDICTION MONTH DISTRIBUTION
# ================================================================

next_basket_episodes[
    "prediction_period"
] = (
    next_basket_episodes[
        "order_date"
    ]
    .dt.to_period(
        "M"
    )
)


monthly_recommendation_summary = (
    next_basket_episodes
    .groupby(
        "prediction_period",
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        unique_customers=(
            "customer",
            "nunique"
        ),

        average_next_basket_size=(
            "next_basket_size",
            "mean"
        ),

        average_repeat_share=(
            "repeat_share",
            "mean"
        ),

        average_new_product_share=(
            "new_product_share",
            "mean"
        ),
    )
)


display(
    monthly_recommendation_summary
)

In [ ]:
# ================================================================
# 57.25 — TARGET PRODUCT SPARSITY
# ================================================================

target_product_frequency = {}


for basket in (
    next_basket_episodes[
        "next_basket_products"
    ]
):

    for product in set(
        basket
    ):

        target_product_frequency[
            product
        ] = (
            target_product_frequency
            .get(
                product,
                0
            )
            +
            1
        )


target_product_frequency = (
    pd.Series(
        target_product_frequency,
        name="target_episodes"
    )
    .sort_values(
        ascending=False
    )
)


target_product_frequency_summary = (
    target_product_frequency
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


display(
    target_product_frequency_summary
)

In [ ]:
# ================================================================
# 57.26 — RARE TARGET PRODUCTS
# ================================================================

rare_target_summary = pd.DataFrame(
    {
        "Threshold": [
            "< 2 target episodes",
            "< 5 target episodes",
            "< 10 target episodes",
            "< 20 target episodes",
        ],

        "Products": [
            (
                target_product_frequency
                < 2
            ).sum(),

            (
                target_product_frequency
                < 5
            ).sum(),

            (
                target_product_frequency
                < 10
            ).sum(),

            (
                target_product_frequency
                < 20
            ).sum(),
        ],
    }
)


rare_target_summary[
    "Share of Target Products (%)"
] = (
    rare_target_summary[
        "Products"
    ]
    /
    len(
        target_product_frequency
    )
    *
    100
)


display(
    rare_target_summary
)

In [ ]:
# ================================================================
# 57.27 — REPEAT SHARE BY ORDER HISTORY
# ================================================================

next_basket_episodes[
    "history_depth_band"
] = pd.cut(
    next_basket_episodes[
        "customer_order_number"
    ],
    bins=[
        0,
        1,
        2,
        3,
        5,
        10,
        np.inf,
    ],
    labels=[
        "After Order 1",
        "After Order 2",
        "After Order 3",
        "After Orders 4–5",
        "After Orders 6–10",
        "After 10+ Orders",
    ]
)


repeat_by_history_depth = (
    next_basket_episodes
    .groupby(
        "history_depth_band",
        observed=True,
        as_index=False
    )
    .agg(
        episodes=(
            "customer",
            "size"
        ),

        average_repeat_share=(
            "repeat_share",
            "mean"
        ),

        average_new_share=(
            "new_product_share",
            "mean"
        ),

        any_repeat_rate=(
            "repeat_product_count",
            lambda values:
                (
                    values
                    > 0
                ).mean()
        ),

        any_new_rate=(
            "new_product_count",
            lambda values:
                (
                    values
                    > 0
                ).mean()
        ),
    )
)


display(
    repeat_by_history_depth
)

## 57.28 Candidate Recommendation Architectures

The first diagnostic stage does not yet select a recommender.

Several increasingly personalised approaches will be compared.

### Baseline 1 — Global Popularity

Recommend the products most frequently purchased across the historical training population.

This establishes how much performance can be achieved without customer personalisation.

### Baseline 2 — Customer Repeat Frequency

Recommend products the customer has purchased most frequently in the past.

This measures the value of replenishment behaviour.

### Candidate 3 — Item-to-Item Affinity

Recommend products that commonly appear with or follow products in the customer's historical basket.

This introduces product-to-product relationships.

### Candidate 4 — Hybrid Recommendation Score

Combine:

- customer purchase history;
- product popularity;
- recency;
- product affinity;
- and possibly repeat versus new-product propensity.

The final recommendation architecture will be selected through chronological out-of-time validation rather than random train/test splitting.

## 57.29 Recommendation Evaluation Metrics

Because the model returns a ranked list of products, ordinary classification accuracy is not sufficient.

The primary metrics will include:

### Hit Rate@K

The proportion of prediction episodes where at least one of the top-K recommendations appears in the actual next basket.

### Recall@K

The proportion of products in the actual next basket that are recovered among the top-K recommendations.

### Precision@K

The proportion of recommended products that appear in the actual next basket.

### Mean Reciprocal Rank

Rewards models that place the first relevant product high in the recommendation list.

### Catalogue Coverage

Measures how much of the product catalogue the recommendation engine is capable of surfacing.

### Cross-Sell Hit Rate

Evaluates performance specifically on products that were new to the customer at the prediction point.

A useful recommendation system should not be judged only by its ability to repeat the same historically purchased items.

## 57.30 Temporal Validation Strategy

Recommendation systems are especially vulnerable to future-information leakage.

Product popularity, customer history and product affinity must all be calculated only from transactions available before each prediction date.

The notebook will therefore use chronological evaluation.

Later validation and Test periods will never contribute:

- product popularity;
- product-pair relationships;
- customer history;
- or candidate-generation statistics

to models evaluated at an earlier point in time.

The final Test period will remain unopened until the recommendation architecture is frozen.

In [ ]:
# ================================================================
# 57.31 — STEP 57 COMPLETION CHECK
# ================================================================

assert (
    len(
        next_basket_episodes
    )
    > 0
)


assert (
    next_basket_episodes[
        "has_next_order_within_90d"
    ]
    .eq(1)
    .all()
)


assert (
    next_basket_episodes[
        "complete_90d_followup"
    ]
    .all()
)


assert (
    next_basket_episodes[
        "days_to_next_order"
    ]
    .le(
        90
    )
    .all()
)


assert (
    next_basket_episodes[
        "next_basket_size"
    ]
    .gt(0)
    .all()
)


print(
    "✓ Step 57 complete."
)

print(
    "✓ Customer-order prediction unit constructed."
)

print(
    "✓ Exact 90-day next-order target constructed."
)

print(
    "✓ Leakage-safe customer product history constructed."
)

print(
    "✓ Repeat and cross-sell target components separated."
)

print(
    "✓ Recommendation target sparsity quantified."
)

print(
    "✓ No recommender has yet been selected."
)

## Step 57 Summary — Recommendation Target Architecture

Step 57 defined the customer-level next-product recommendation problem and examined the structure of the next observed customer basket.

### Next Baskets Are Multi-Product

The recommendation population contained **21,431 prediction episodes**.

The next customer order contained on average approximately:

> **18.9 distinct products**

with a median of:

> **14 products**

The 90th percentile contained approximately **42 products**.

Recommendation performance should therefore be evaluated at multiple list depths, particularly **Top-5, Top-10 and Top-20**.

### Repeat and Discovery Behaviour Are Both Material

The average next basket contained approximately:

- **9.3 previously purchased products**
- **9.6 new-to-customer products**

Average basket composition was therefore approximately:

- **52.6% repeat**
- **47.4% new products**

Approximately **88.3%** of next baskets contained at least one repeat product.

Approximately **82.2%** contained at least one new-to-customer product.

Only approximately **17.8%** of baskets were entirely repeat purchases and **11.7%** were entirely new products.

The recommendation problem is therefore fundamentally hybrid.

### Customer Maturity Changes Product Behaviour

Repeat purchasing increases substantially with customer order history.

Average repeat-product share increased from approximately:

- **25.5% after the first order**

to:

- **68.9% after more than ten orders**

New customers are therefore more discovery-oriented, while mature customers become increasingly replenishment-oriented.

This suggests that the eventual recommendation architecture should allow recommendation-channel weighting to vary with customer history depth.

### Catalogue Demand Is Broad

The catalogue is not dominated by a very small number of products.

The top:

- 10 products represented approximately **3.6%** of product-order occurrences;
- 100 represented approximately **19.3%**;
- 500 represented approximately **52.0%**.

Global popularity therefore provides a necessary benchmark but is unlikely to capture the full recommendation opportunity.

### The Product Target Contains a Long Tail

Approximately **4,456 products** appeared as next-basket targets.

Around:

- **25.3%** appeared in fewer than 10 target episodes;
- **38.7%** appeared in fewer than 20.

Sparse products will therefore require fallback recommendation logic when direct product-affinity evidence is limited.

## Architecture Decision

Notebook 6 will develop a:

> **Hybrid Next-Best-Product Recommender**

with two distinct recommendation channels:

### Repeat / Replenishment Channel

Ranks products the customer has previously purchased.

### Discovery / Cross-Sell Channel

Ranks products that are new to the customer.

The two channels will later be combined into a single customer recommendation list.

The final weighting will be determined through chronological validation rather than assumed in advance.


## Step 58 — Temporal Split and Recommendation Baselines

Step 58 establishes chronological recommendation populations and simple benchmark recommenders.

The initial dataset months are treated as a burn-in period to reduce extreme left-truncation of customer and product histories.

The designated periods are:

- **Train:** March–November 2010
- **Validation:** March–April 2011
- **Test:** July–August 2011

The final Test population will remain unopened until the recommendation architecture has been frozen.

Two baseline recommendation strategies are evaluated first:

1. **Global Product Popularity**
2. **Customer Repeat Frequency**

These establish the performance that must be exceeded by more sophisticated product-affinity and hybrid recommendation approaches.

In [ ]:
# ================================================================
# 58.2 — TEMPORAL PERIOD DEFINITIONS
# ================================================================

REC_TRAIN_START = pd.Period(
    "2010-03",
    freq="M"
)

REC_TRAIN_END = pd.Period(
    "2010-11",
    freq="M"
)


REC_VALIDATION_START = pd.Period(
    "2011-03",
    freq="M"
)

REC_VALIDATION_END = pd.Period(
    "2011-04",
    freq="M"
)


REC_TEST_START = pd.Period(
    "2011-07",
    freq="M"
)

REC_TEST_END = pd.Period(
    "2011-08",
    freq="M"
)

In [ ]:
# ================================================================
# 58.3 — TEMPORAL POPULATIONS
# ================================================================

recommendation_train = (
    next_basket_episodes.loc[
        next_basket_episodes[
            "prediction_period"
        ]
        .between(
            REC_TRAIN_START,
            REC_TRAIN_END
        )
    ]
    .copy()
)


recommendation_validation = (
    next_basket_episodes.loc[
        next_basket_episodes[
            "prediction_period"
        ]
        .between(
            REC_VALIDATION_START,
            REC_VALIDATION_END
        )
    ]
    .copy()
)


recommendation_test = (
    next_basket_episodes.loc[
        next_basket_episodes[
            "prediction_period"
        ]
        .between(
            REC_TEST_START,
            REC_TEST_END
        )
    ]
    .copy()
)

In [ ]:
# ================================================================
# 58.4 — DEVELOPMENT POPULATION SUMMARY
# ================================================================

recommendation_split_summary = pd.DataFrame(
    [
        {
            "Dataset":
                "Train",

            "Episodes":
                len(
                    recommendation_train
                ),

            "Customers":
                recommendation_train[
                    "customer"
                ].nunique(),

            "Start":
                recommendation_train[
                    "prediction_period"
                ].min(),

            "End":
                recommendation_train[
                    "prediction_period"
                ].max(),

            "Median Basket Size":
                recommendation_train[
                    "next_basket_size"
                ].median(),

            "Average Repeat Share":
                recommendation_train[
                    "repeat_share"
                ].mean(),

            "Average New Share":
                recommendation_train[
                    "new_product_share"
                ].mean(),
        },

        {
            "Dataset":
                "Validation",

            "Episodes":
                len(
                    recommendation_validation
                ),

            "Customers":
                recommendation_validation[
                    "customer"
                ].nunique(),

            "Start":
                recommendation_validation[
                    "prediction_period"
                ].min(),

            "End":
                recommendation_validation[
                    "prediction_period"
                ].max(),

            "Median Basket Size":
                recommendation_validation[
                    "next_basket_size"
                ].median(),

            "Average Repeat Share":
                recommendation_validation[
                    "repeat_share"
                ].mean(),

            "Average New Share":
                recommendation_validation[
                    "new_product_share"
                ].mean(),
        },
    ]
)


display(
    recommendation_split_summary
)

In [ ]:
# ================================================================
# 58.5 — TEST GOVERNANCE
# ================================================================

RECOMMENDATION_TEST_SET_OPENED = False


print(
    f"Test episodes reserved: "
    f"{len(recommendation_test):,}"
)

print(
    f"Test prediction period: "
    f"{recommendation_test['prediction_period'].min()} "
    f"to "
    f"{recommendation_test['prediction_period'].max()}"
)

print(
    "Test next-basket outcomes remain intentionally unopened."
)

In [ ]:
# ================================================================
# 58.6 — SINGLE-EPISODE RECOMMENDATION METRICS
# ================================================================

def recommendation_episode_metrics(
    actual_products,
    recommended_products,
    k
):

    actual = set(
        actual_products
    )


    recommended = list(
        recommended_products
    )[:k]


    recommended_set = set(
        recommended
    )


    hits = (
        actual
        &
        recommended_set
    )


    hit_rate = int(
        len(hits) > 0
    )


    recall = (
        len(hits)
        /
        len(actual)
        if len(actual) > 0
        else np.nan
    )


    precision = (
        len(hits)
        /
        k
        if k > 0
        else np.nan
    )


    reciprocal_rank = 0.0


    for rank, product in enumerate(
        recommended,
        start=1
    ):

        if product in actual:

            reciprocal_rank = (
                1.0
                /
                rank
            )

            break


    return {
        "hit_rate":
            hit_rate,

        "recall":
            recall,

        "precision":
            precision,

        "reciprocal_rank":
            reciprocal_rank,
    }

In [ ]:
# ================================================================
# 58.7 — RECOMMENDER EVALUATION
# ================================================================

def evaluate_recommendation_lists(
    actual_baskets,
    recommendation_lists,
    model_name,
    ks=(5, 10, 20)
):

    output = []


    for k in ks:

        episode_results = []


        for (
            actual,
            recommended
        ) in zip(
            actual_baskets,
            recommendation_lists
        ):

            episode_results.append(
                recommendation_episode_metrics(
                    actual,
                    recommended,
                    k
                )
            )


        episode_results = pd.DataFrame(
            episode_results
        )


        output.append(
            {
                "Model":
                    model_name,

                "K":
                    k,

                "Hit Rate":
                    episode_results[
                        "hit_rate"
                    ].mean(),

                "Recall":
                    episode_results[
                        "recall"
                    ].mean(),

                "Precision":
                    episode_results[
                        "precision"
                    ].mean(),

                "MRR":
                    episode_results[
                        "reciprocal_rank"
                    ].mean(),
            }
        )


    return pd.DataFrame(
        output
    )

## 58.8 — Product history before each prediction
We already have historical_products, but for repeat ranking we need purchase frequency and recency.

Build customer-product transactions from all lines available up to each prediction point.

For the first baseline, we'll make repeat recommendations episode-by-episode.

In [ ]:
# ================================================================
# 58.8 — CUSTOMER PRODUCT PURCHASE HISTORY TABLE
# ================================================================

customer_product_orders = (
    recommendation_lines[
        [
            "customer",
            "invoice",
            "product",
            "date",
        ]
    ]
    .drop_duplicates(
        [
            "customer",
            "invoice",
            "product",
        ]
    )
    .sort_values(
        [
            "customer",
            "date",
            "invoice",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Customer-product-order rows: "
    f"{len(customer_product_orders):,}"
)

In [ ]:
# ================================================================
# 58.9 — TEMPORAL POPULARITY RECOMMENDER
# ================================================================

def temporal_popularity_recommendations(
    episodes,
    transaction_lines,
    top_n=20,
    exclude_history=False
):

    recommendation_lists = []


    for row in episodes.itertuples(
        index=False
    ):

        cutoff = row.order_date


        historical_transactions = (
            transaction_lines.loc[
                transaction_lines[
                    "date"
                ]
                <=
                cutoff
            ]
        )


        popularity = (
            historical_transactions
            .groupby(
                "product"
            )[
                "invoice"
            ]
            .nunique()
            .sort_values(
                ascending=False
            )
        )


        ranked_products = list(
            popularity.index
        )


        if exclude_history:

            historical_customer_products = set(
                row.historical_products
            )


            ranked_products = [
                product
                for product in ranked_products
                if product
                not in
                historical_customer_products
            ]


        recommendation_lists.append(
            ranked_products[
                :top_n
            ]
        )


    return recommendation_lists

### 58.10 — Repeat-frequency recommender
Rank customer products by:
1. purchase frequency;
2. most recent purchase as tie-breaker.

In [ ]:
# ================================================================
# 58.10 — CUSTOMER REPEAT-FREQUENCY RECOMMENDER
# ================================================================

def repeat_frequency_recommendations(
    episodes,
    customer_product_history,
    top_n=20
):

    recommendation_lists = []


    for row in episodes.itertuples(
        index=False
    ):

        history = (
            customer_product_history.loc[
                (
                    customer_product_history[
                        "customer"
                    ]
                    ==
                    row.customer
                )
                &
                (
                    customer_product_history[
                        "date"
                    ]
                    <=
                    row.order_date
                )
            ]
            .copy()
        )


        if history.empty:

            recommendation_lists.append(
                []
            )

            continue


        product_summary = (
            history
            .groupby(
                "product",
                as_index=False
            )
            .agg(
                purchase_orders=(
                    "invoice",
                    "nunique"
                ),

                last_purchase=(
                    "date",
                    "max"
                ),
            )
            .sort_values(
                [
                    "purchase_orders",
                    "last_purchase",
                ],
                ascending=[
                    False,
                    False,
                ]
            )
        )


        recommendation_lists.append(
            product_summary[
                "product"
            ]
            .head(
                top_n
            )
            .tolist()
        )


    return recommendation_lists

In [ ]:
# ================================================================
# 58.11 — GLOBAL POPULARITY BASELINE
# ================================================================

validation_popularity_recommendations = (
    temporal_popularity_recommendations(
        episodes=
            recommendation_validation,

        transaction_lines=
            customer_product_orders,

        top_n=20,

        exclude_history=False
    )
)


print(
    "✓ Temporal popularity recommendations generated."
)

In [ ]:
# ================================================================
# 58.12 — POPULARITY CROSS-SELL BASELINE
# ================================================================

validation_cross_sell_popularity = (
    temporal_popularity_recommendations(
        episodes=
            recommendation_validation,

        transaction_lines=
            customer_product_orders,

        top_n=20,

        exclude_history=True
    )
)


print(
    "✓ Cross-sell popularity recommendations generated."
)

In [ ]:
# ================================================================
# 58.13 — REPEAT-FREQUENCY BASELINE
# ================================================================

validation_repeat_recommendations = (
    repeat_frequency_recommendations(
        episodes=
            recommendation_validation,

        customer_product_history=
            customer_product_orders,

        top_n=20
    )
)


print(
    "✓ Customer repeat-frequency recommendations generated."
)

In [ ]:
# ================================================================
# 58.14 — GENERAL NEXT-BASKET BASELINES
# ================================================================

popularity_baseline_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_validation[
                "next_basket_products"
            ],

        recommendation_lists=
            validation_popularity_recommendations,

        model_name=
            "Global Popularity"
    )
)


repeat_baseline_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_validation[
                "next_basket_products"
            ],

        recommendation_lists=
            validation_repeat_recommendations,

        model_name=
            "Customer Repeat Frequency"
    )
)


general_recommendation_baselines = (
    pd.concat(
        [
            popularity_baseline_results,
            repeat_baseline_results,
        ],
        ignore_index=True
    )
)


display(
    general_recommendation_baselines
)

In [ ]:
# ================================================================
# 58.15 — CROSS-SELL POPULARITY PERFORMANCE
# ================================================================

cross_sell_popularity_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_validation[
                "new_products"
            ],

        recommendation_lists=
            validation_cross_sell_popularity,

        model_name=
            "Popularity Cross-Sell"
    )
)


display(
    cross_sell_popularity_results
)

In [ ]:
# ================================================================
# 58.16 — CROSS-SELL ELIGIBLE POPULATION
# ================================================================

cross_sell_mask = (
    recommendation_validation[
        "new_product_count"
    ]
    > 0
)


cross_sell_eligible_validation = (
    recommendation_validation.loc[
        cross_sell_mask
    ]
    .copy()
)


cross_sell_eligible_recommendations = [
    recommendation
    for recommendation, eligible
    in zip(
        validation_cross_sell_popularity,
        cross_sell_mask
    )
    if eligible
]


cross_sell_eligible_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            cross_sell_eligible_validation[
                "new_products"
            ],

        recommendation_lists=
            cross_sell_eligible_recommendations,

        model_name=
            "Popularity Cross-Sell — Eligible Episodes"
    )
)


display(
    cross_sell_eligible_results
)

In [ ]:
# ================================================================
# 58.17 — REPEAT-ELIGIBLE POPULATION
# ================================================================

repeat_mask = (
    recommendation_validation[
        "repeat_product_count"
    ]
    > 0
)


repeat_eligible_validation = (
    recommendation_validation.loc[
        repeat_mask
    ]
    .copy()
)


repeat_eligible_recommendations = [
    recommendation
    for recommendation, eligible
    in zip(
        validation_repeat_recommendations,
        repeat_mask
    )
    if eligible
]


repeat_eligible_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            repeat_eligible_validation[
                "repeat_products"
            ],

        recommendation_lists=
            repeat_eligible_recommendations,

        model_name=
            "Repeat Frequency — Eligible Episodes"
    )
)


display(
    repeat_eligible_results
)

In [ ]:
# ================================================================
# 58.18 — RECOMMENDATION CATALOGUE COVERAGE
# ================================================================

def recommendation_catalogue_coverage(
    recommendation_lists,
    catalogue_products,
    k
):

    recommended_products = set()


    for recommendations in (
        recommendation_lists
    ):

        recommended_products.update(
            recommendations[:k]
        )


    catalogue_products = set(
        catalogue_products
    )


    return (
        len(
            recommended_products
        )
        /
        len(
            catalogue_products
        )
        if len(
            catalogue_products
        ) > 0
        else np.nan
    )

In [ ]:
# ================================================================
# 58.19 — BASELINE CATALOGUE COVERAGE
# ================================================================

validation_end_date = (
    recommendation_validation[
        "order_date"
    ]
    .max()
)


available_validation_catalogue = (
    customer_product_orders.loc[
        customer_product_orders[
            "date"
        ]
        <=
        validation_end_date,
        "product"
    ]
    .unique()
)


baseline_coverage_records = []


for model_name, recommendations in [
    (
        "Global Popularity",
        validation_popularity_recommendations
    ),

    (
        "Customer Repeat Frequency",
        validation_repeat_recommendations
    ),

    (
        "Popularity Cross-Sell",
        validation_cross_sell_popularity
    ),
]:

    for k in [
        5,
        10,
        20,
    ]:

        baseline_coverage_records.append(
            {
                "Model":
                    model_name,

                "K":
                    k,

                "Catalogue Coverage":
                    recommendation_catalogue_coverage(
                        recommendation_lists=
                            recommendations,

                        catalogue_products=
                            available_validation_catalogue,

                        k=
                            k
                    ),
            }
        )


baseline_catalogue_coverage = pd.DataFrame(
    baseline_coverage_records
)


baseline_catalogue_coverage[
    "Catalogue Coverage (%)"
] = (
    baseline_catalogue_coverage[
        "Catalogue Coverage"
    ]
    *
    100
)


display(
    baseline_catalogue_coverage
)

In [ ]:
# ================================================================
# 58.20 — REPEAT PERFORMANCE BY HISTORY DEPTH
# ================================================================

history_depth_performance_records = []


for history_band in (
    recommendation_validation[
        "history_depth_band"
    ]
    .dropna()
    .unique()
):

    mask = (
        recommendation_validation[
            "history_depth_band"
        ]
        ==
        history_band
    )


    subset = (
        recommendation_validation.loc[
            mask
        ]
    )


    subset_recommendations = [
        recommendations
        for recommendations, keep
        in zip(
            validation_repeat_recommendations,
            mask
        )
        if keep
    ]


    metrics = (
        evaluate_recommendation_lists(
            actual_baskets=
                subset[
                    "next_basket_products"
                ],

            recommendation_lists=
                subset_recommendations,

            model_name=
                str(
                    history_band
                ),

            ks=(
                10,
            )
        )
    )


    history_depth_performance_records.append(
        metrics.iloc[0]
        .to_dict()
    )


repeat_performance_by_history_depth = (
    pd.DataFrame(
        history_depth_performance_records
    )
)


display(
    repeat_performance_by_history_depth
)

In [ ]:
# ================================================================
# 58.21 — TEST GOVERNANCE CHECK
# ================================================================

assert RECOMMENDATION_TEST_SET_OPENED is False


print(
    "✓ Recommendation Test outcomes remain unopened."
)

In [ ]:
# ================================================================
# 58.22 — STEP 58 COMPLETE
# ================================================================

assert len(
    recommendation_train
) > 0

assert len(
    recommendation_validation
) > 0

assert len(
    recommendation_test
) > 0

assert (
    RECOMMENDATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 58 complete."
)

print(
    "✓ Chronological recommendation populations created."
)

print(
    "✓ Global popularity baseline evaluated."
)

print(
    "✓ Customer repeat-frequency baseline evaluated."
)

print(
    "✓ Cross-sell popularity baseline evaluated."
)

print(
    "✓ Catalogue coverage measured."
)

print(
    "✓ Recommendation Test remains unopened."
)

## Step 58 Summary — Recommendation Baselines

Step 58 established chronological recommendation populations and benchmark recommendation strategies.

### Validation Became Slightly More Replenishment-Oriented

Average repeat-product share increased from approximately **51.9% in Train** to approximately **56.0% in Validation**.

The recommendation environment therefore changes somewhat over time, although both repeat and discovery behaviour remain economically relevant.

### Customer Purchase History Is a Strong Recommendation Signal

Customer Repeat Frequency substantially outperformed global popularity.

At Top-10, the repeat-frequency recommender achieved approximately:

- **Hit Rate: 70.6%**
- **Recall: 18.7%**
- **Precision: 20.2%**
- **MRR: 0.413**

Among episodes that actually contained repeat products, Top-10 Hit Rate increased to approximately **77.9%**.

Repeat-frequency recommendation therefore represents a strong replenishment benchmark that later models must exceed or complement.

### Global Popularity Is Much Weaker

At Top-10, global popularity achieved only approximately:

- **Hit Rate: 43.4%**
- **Recall: 4.9%**
- **Precision: 6.9%**

It also exposed only a very small proportion of the available product catalogue.

Global popularity is therefore useful primarily as a fallback rather than as the primary personalized recommendation strategy.

### Cross-Sell Remains the Harder Problem

Among episodes containing at least one new-to-customer product, the popularity-based cross-sell recommender achieved approximately:

- **Top-10 Hit Rate: 15.6%**
- **Top-10 Recall: 2.0%**
- **Top-10 Precision: 2.0%**
- **MRR: 0.065**

The discovery channel therefore contains substantially more room for improvement than the replenishment channel.

### Repeat Recommendation Performance Depends on Customer History

Repeat-frequency recommendation generally improved as customer order history deepened.

Top-10 Hit Rate was approximately:

- **59.1% after the first order**
- **69–70% after orders 2–5**
- **75.1% after orders 6–10**

Performance did not continue increasing monotonically for the most mature customers, suggesting that very large historical product repertoires eventually make product selection more difficult.

## Baseline Decision

The next modelling stage should preserve:

> **Customer Repeat Frequency**

as a strong replenishment benchmark.

The largest opportunity for additional personalization is:

> **item-to-item affinity for new-product discovery**

The final system should therefore develop separate repeat and cross-sell candidate scores before combining them into a hybrid recommendation list.

## Step 59 — Item-to-Item Affinity and Hybrid Recommendation

Step 59 introduces product-level personalization beyond simple customer repeat history.

Product affinity is estimated from historical order baskets available before each Validation month.

The primary objective is to improve discovery of products that are new to the customer while preserving the strong replenishment performance already provided by customer repeat frequency.

Three components will be evaluated:

1. **Repeat Frequency**
2. **Item-to-Item Affinity**
3. **Hybrid Repeat + Discovery Recommendation**

Affinity matrices are created using only historical baskets available before the relevant prediction month.

The Jul–Aug 2011 performance holdout remains unused for recommendation evaluation.

In [ ]:
# ================================================================
# 59.2 — HISTORICAL ORDER BASKETS
# ================================================================

historical_baskets = (
    recommendation_lines
    .groupby(
        [
            "invoice",
            "customer",
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        products=(
            "product",
            lambda values:
                tuple(
                    sorted(
                        set(values)
                    )
                )
        ),
    )
)


historical_baskets = (
    historical_baskets
    .sort_values(
        "order_date"
    )
    .reset_index(drop=True)
)


print(
    f"Historical baskets: "
    f"{len(historical_baskets):,}"
)

In [ ]:
# ================================================================
# 59.3 — ITEM AFFINITY MATRIX
# ================================================================

from collections import (
    Counter,
    defaultdict,
)


def build_item_affinity(
    baskets,
    minimum_item_orders=5,
    minimum_pair_orders=2
):

    item_count = Counter()
    pair_count = defaultdict(Counter)


    for products in baskets:

        products = list(
            set(products)
        )


        for item in products:

            item_count[
                item
            ] += 1


        for item_a in products:

            for item_b in products:

                if (
                    item_a
                    ==
                    item_b
                ):
                    continue


                pair_count[
                    item_a
                ][
                    item_b
                ] += 1


    affinity = {}


    for item_a, neighbours in (
        pair_count.items()
    ):

        if (
            item_count[
                item_a
            ]
            <
            minimum_item_orders
        ):
            continue


        scored_neighbours = []


        for (
            item_b,
            cooccurrence
        ) in (
            neighbours.items()
        ):

            if (
                cooccurrence
                <
                minimum_pair_orders
            ):
                continue


            if (
                item_count[
                    item_b
                ]
                <
                minimum_item_orders
            ):
                continue


            similarity = (
                cooccurrence
                /
                np.sqrt(
                    item_count[
                        item_a
                    ]
                    *
                    item_count[
                        item_b
                    ]
                )
            )


            scored_neighbours.append(
                (
                    item_b,
                    similarity,
                    cooccurrence,
                )
            )


        scored_neighbours.sort(
            key=lambda x:
                (
                    x[1],
                    x[2],
                ),
            reverse=True
        )


        affinity[
            item_a
        ] = scored_neighbours


    return (
        affinity,
        item_count,
    )

In [ ]:
# ================================================================
# 59.4 — VALIDATION AFFINITY SNAPSHOTS
# ================================================================

validation_month_starts = {
    pd.Period(
        "2011-03",
        freq="M"
    ):
        pd.Timestamp(
            "2011-03-01"
        ),

    pd.Period(
        "2011-04",
        freq="M"
    ):
        pd.Timestamp(
            "2011-04-01"
        ),
}


affinity_snapshots = {}
popularity_snapshots = {}


for period, cutoff in (
    validation_month_starts.items()
):

    past_baskets = (
        historical_baskets.loc[
            historical_baskets[
                "order_date"
            ]
            <
            cutoff
        ]
        .copy()
    )


    affinity, item_count = (
        build_item_affinity(
            baskets=
                past_baskets[
                    "products"
                ],

            minimum_item_orders=5,

            minimum_pair_orders=2
        )
    )


    affinity_snapshots[
        period
    ] = affinity


    popularity_snapshots[
        period
    ] = [
        item
        for item, count
        in sorted(
            item_count.items(),
            key=lambda x:
                x[1],
            reverse=True
        )
    ]


    print(
        period,
        "baskets:",
        len(
            past_baskets
        ),
        "affinity source products:",
        len(
            affinity
        )
    )

In [ ]:
# ================================================================
# 59.5 — CUSTOMER-PRODUCT HISTORY SUMMARY
# ================================================================

def customer_product_summary_at_date(
    customer,
    cutoff,
    history_table
):

    history = (
        history_table.loc[
            (
                history_table[
                    "customer"
                ]
                ==
                customer
            )
            &
            (
                history_table[
                    "date"
                ]
                <=
                cutoff
            )
        ]
        .copy()
    )


    if history.empty:

        return pd.DataFrame(
            columns=[
                "product",
                "purchase_orders",
                "last_purchase",
            ]
        )


    summary = (
        history
        .groupby(
            "product",
            as_index=False
        )
        .agg(
            purchase_orders=(
                "invoice",
                "nunique"
            ),

            last_purchase=(
                "date",
                "max"
            ),
        )
    )


    return summary

In [ ]:
# ================================================================
# 59.6 — CUSTOMER AFFINITY SCORES
# ================================================================

def affinity_scores_for_customer(
    product_history_summary,
    affinity_dictionary,
    exclude_history=False
):

    scores = defaultdict(float)


    if (
        product_history_summary.empty
    ):

        return {}


    historical_products = set(
        product_history_summary[
            "product"
        ]
    )


    for row in (
        product_history_summary
        .itertuples(
            index=False
        )
    ):

        source_product = (
            row.product
        )


        source_weight = np.log1p(
            row.purchase_orders
        )


        neighbours = (
            affinity_dictionary
            .get(
                source_product,
                []
            )
        )


        for (
            candidate,
            similarity,
            pair_count
        ) in neighbours:

            if (
                exclude_history
                and
                candidate
                in
                historical_products
            ):
                continue


            scores[
                candidate
            ] += (
                source_weight
                *
                similarity
            )


    return dict(
        scores
    )

In [ ]:
# ================================================================
# 59.7 — AFFINITY RECOMMENDATION LIST
# ================================================================

def rank_affinity_recommendations(
    product_history_summary,
    affinity_dictionary,
    popularity_list,
    top_n=20,
    exclude_history=False
):

    scores = (
        affinity_scores_for_customer(
            product_history_summary=
                product_history_summary,

            affinity_dictionary=
                affinity_dictionary,

            exclude_history=
                exclude_history
        )
    )


    ranked = [
        product
        for product, score
        in sorted(
            scores.items(),
            key=lambda x:
                x[1],
            reverse=True
        )
    ]


    historical_products = set(
        product_history_summary[
            "product"
        ]
    )


    for product in popularity_list:

        if (
            exclude_history
            and
            product
            in
            historical_products
        ):
            continue


        if (
            product
            not in
            ranked
        ):

            ranked.append(
                product
            )


        if (
            len(ranked)
            >=
            top_n
        ):
            break


    return ranked[
        :top_n
    ]

In [ ]:
# ================================================================
# 59.8 — GENERATE AFFINITY RECOMMENDATIONS
# ================================================================

validation_affinity_lists = []
validation_cross_sell_affinity_lists = []


for row in (
    recommendation_validation
    .itertuples(
        index=False
    )
):

    period = (
        row.prediction_period
    )


    product_history = (
        customer_product_summary_at_date(
            customer=
                row.customer,

            cutoff=
                row.order_date,

            history_table=
                customer_product_orders
        )
    )


    affinity_dictionary = (
        affinity_snapshots[
            period
        ]
    )


    popularity_list = (
        popularity_snapshots[
            period
        ]
    )


    validation_affinity_lists.append(
        rank_affinity_recommendations(
            product_history_summary=
                product_history,

            affinity_dictionary=
                affinity_dictionary,

            popularity_list=
                popularity_list,

            top_n=20,

            exclude_history=False
        )
    )


    validation_cross_sell_affinity_lists.append(
        rank_affinity_recommendations(
            product_history_summary=
                product_history,

            affinity_dictionary=
                affinity_dictionary,

            popularity_list=
                popularity_list,

            top_n=20,

            exclude_history=True
        )
    )


print(
    "✓ Validation affinity recommendations generated."
)

In [ ]:
# ================================================================
# 59.9 — GENERAL AFFINITY PERFORMANCE
# ================================================================

affinity_general_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_validation[
                "next_basket_products"
            ],

        recommendation_lists=
            validation_affinity_lists,

        model_name=
            "Item-to-Item Affinity"
    )
)


display(
    affinity_general_results
)

In [ ]:
# ================================================================
# 59.10 — AFFINITY CROSS-SELL PERFORMANCE
# ================================================================

affinity_cross_sell_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_validation[
                "new_products"
            ],

        recommendation_lists=
            validation_cross_sell_affinity_lists,

        model_name=
            "Item Affinity Cross-Sell"
    )
)


display(
    affinity_cross_sell_results
)

In [ ]:
# ================================================================
# 59.11 — CROSS-SELL-ELIGIBLE AFFINITY
# ================================================================

affinity_cross_sell_eligible_lists = [
    recommendations
    for recommendations, eligible
    in zip(
        validation_cross_sell_affinity_lists,
        cross_sell_mask
    )
    if eligible
]


affinity_cross_sell_eligible_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            cross_sell_eligible_validation[
                "new_products"
            ],

        recommendation_lists=
            affinity_cross_sell_eligible_lists,

        model_name=
            "Affinity Cross-Sell — Eligible Episodes"
    )
)


display(
    affinity_cross_sell_eligible_results
)

### 59.12 — Build hybrid list
Now combine:
- repeat frequency list;
- affinity discovery list.

Do not use the all-period Step 57 maturity shares as weights.

Instead, we'll start with transparent fixed allocations and compare them.

For Top-20 we test:
- 50/50 → 10 repeat + 10 discovery
- 60/40 → 12 repeat + 8 discovery
- 70/30 → 14 repeat + 6 discovery

In [ ]:
# ================================================================
# 59.12 — HYBRID LIST BUILDER
# ================================================================

def build_hybrid_list(
    repeat_list,
    discovery_list,
    top_n=20,
    repeat_fraction=0.60
):

    repeat_slots = int(
        round(
            top_n
            *
            repeat_fraction
        )
    )


    discovery_slots = (
        top_n
        -
        repeat_slots
    )


    output = []


    for product in (
        repeat_list[
            :repeat_slots
        ]
    ):

        if (
            product
            not in
            output
        ):

            output.append(
                product
            )


    for product in (
        discovery_list[
            :discovery_slots
        ]
    ):

        if (
            product
            not in
            output
        ):

            output.append(
                product
            )


    # Fill unused slots while preserving channel order
    for source in [
        repeat_list,
        discovery_list,
    ]:

        for product in source:

            if (
                product
                not in
                output
            ):

                output.append(
                    product
                )


            if (
                len(output)
                >=
                top_n
            ):

                return output[
                    :top_n
                ]


    return output[
        :top_n
    ]

In [ ]:
# ================================================================
# 59.13 — HYBRID CANDIDATES
# ================================================================

hybrid_weight_candidates = {
    "Hybrid 50R / 50D":
        0.50,

    "Hybrid 60R / 40D":
        0.60,

    "Hybrid 70R / 30D":
        0.70,
}


validation_hybrid_lists = {}


for (
    model_name,
    repeat_fraction
) in (
    hybrid_weight_candidates.items()
):

    lists = []


    for (
        repeat_list,
        discovery_list
    ) in zip(
        validation_repeat_recommendations,
        validation_cross_sell_affinity_lists
    ):

        lists.append(
            build_hybrid_list(
                repeat_list=
                    repeat_list,

                discovery_list=
                    discovery_list,

                top_n=20,

                repeat_fraction=
                    repeat_fraction
            )
        )


    validation_hybrid_lists[
        model_name
    ] = lists

In [ ]:
# ================================================================
# 59.14 — HYBRID PERFORMANCE
# ================================================================

hybrid_result_frames = []


for (
    model_name,
    recommendations
) in (
    validation_hybrid_lists.items()
):

    result = (
        evaluate_recommendation_lists(
            actual_baskets=
                recommendation_validation[
                    "next_basket_products"
                ],

            recommendation_lists=
                recommendations,

            model_name=
                model_name
        )
    )


    hybrid_result_frames.append(
        result
    )


hybrid_recommendation_results = pd.concat(
    hybrid_result_frames,
    ignore_index=True
)


display(
    hybrid_recommendation_results
)

In [ ]:
# ================================================================
# 59.15 — FULL VALIDATION COMPARISON
# ================================================================

recommendation_model_comparison = (
    pd.concat(
        [
            general_recommendation_baselines,
            affinity_general_results,
            hybrid_recommendation_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "K",
            "Hit Rate",
            "Recall",
        ],
        ascending=[
            True,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


display(
    recommendation_model_comparison
)

In [ ]:
# ================================================================
# 59.16 — CROSS-SELL COMPARISON
# ================================================================

cross_sell_model_comparison = (
    pd.concat(
        [
            cross_sell_eligible_results,
            affinity_cross_sell_eligible_results,
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "K",
            "Hit Rate",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .reset_index(drop=True)
)


display(
    cross_sell_model_comparison
)

In [ ]:
# ================================================================
# 59.17 — HYBRID CATALOGUE COVERAGE
# ================================================================

hybrid_coverage_records = []


for (
    model_name,
    recommendations
) in (
    validation_hybrid_lists.items()
):

    for k in [
        5,
        10,
        20,
    ]:

        coverage = (
            recommendation_catalogue_coverage(
                recommendation_lists=
                    recommendations,

                catalogue_products=
                    available_validation_catalogue,

                k=
                    k
            )
        )


        hybrid_coverage_records.append(
            {
                "Model":
                    model_name,

                "K":
                    k,

                "Catalogue Coverage":
                    coverage,

                "Catalogue Coverage (%)":
                    coverage
                    *
                    100,
            }
        )


hybrid_catalogue_coverage = pd.DataFrame(
    hybrid_coverage_records
)


display(
    hybrid_catalogue_coverage
)

In [ ]:
# ================================================================
# 59.18 — CHANNEL HIT CONTRIBUTION
# ================================================================

def hybrid_channel_hit_summary(
    episodes,
    repeat_lists,
    discovery_lists,
    k_repeat,
    k_discovery
):

    repeat_hit = []
    discovery_hit = []


    for (
        row,
        repeat_list,
        discovery_list
    ) in zip(
        episodes.itertuples(
            index=False
        ),
        repeat_lists,
        discovery_lists
    ):

        actual_repeat = set(
            row.repeat_products
        )

        actual_new = set(
            row.new_products
        )


        repeat_hit.append(
            int(
                len(
                    actual_repeat
                    &
                    set(
                        repeat_list[
                            :k_repeat
                        ]
                    )
                )
                > 0
            )
        )


        discovery_hit.append(
            int(
                len(
                    actual_new
                    &
                    set(
                        discovery_list[
                            :k_discovery
                        ]
                    )
                )
                > 0
            )
        )


    return pd.DataFrame(
        {
            "Metric": [
                "Repeat Channel Hit Rate",
                "Discovery Channel Hit Rate",
            ],

            "Value": [
                np.mean(
                    repeat_hit
                ),

                np.mean(
                    discovery_hit
                ),
            ],
        }
    )

In [ ]:
hybrid_60_channel_summary = (
    hybrid_channel_hit_summary(
        episodes=
            recommendation_validation,

        repeat_lists=
            validation_repeat_recommendations,

        discovery_lists=
            validation_cross_sell_affinity_lists,

        k_repeat=12,

        k_discovery=8
    )
)


display(
    hybrid_60_channel_summary
)

In [ ]:
# ================================================================
# 59.19 — HYBRID PERFORMANCE BY CUSTOMER MATURITY
# ================================================================

hybrid_history_depth_records = []


for model_name, recommendations in (
    validation_hybrid_lists.items()
):

    for history_band in (
        recommendation_validation[
            "history_depth_band"
        ]
        .dropna()
        .unique()
    ):

        mask = (
            recommendation_validation[
                "history_depth_band"
            ]
            ==
            history_band
        )


        subset = (
            recommendation_validation.loc[
                mask
            ]
        )


        subset_recommendations = [
            recommendation
            for recommendation, keep
            in zip(
                recommendations,
                mask
            )
            if keep
        ]


        result = (
            evaluate_recommendation_lists(
                actual_baskets=
                    subset[
                        "next_basket_products"
                    ],

                recommendation_lists=
                    subset_recommendations,

                model_name=
                    model_name,

                ks=(
                    10,
                    20,
                )
            )
        )


        result[
            "History Depth"
        ] = str(
            history_band
        )


        hybrid_history_depth_records.append(
            result
        )


hybrid_performance_by_history_depth = (
    pd.concat(
        hybrid_history_depth_records,
        ignore_index=True
    )
)


display(
    hybrid_performance_by_history_depth
)

In [ ]:
# ================================================================
# 59.20 — GOVERNANCE
# ================================================================

assert (
    RECOMMENDATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Performance holdout remains unused."
)

In [ ]:
# ================================================================
# 59.21 — STEP 59 COMPLETE
# ================================================================

assert len(
    affinity_snapshots
) == 2


assert (
    len(
        validation_affinity_lists
    )
    ==
    len(
        recommendation_validation
    )
)


assert (
    RECOMMENDATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 59 complete."
)

print(
    "✓ Leakage-safe monthly item-affinity snapshots built."
)

print(
    "✓ Affinity recommender evaluated."
)

print(
    "✓ Cross-sell affinity evaluated."
)

print(
    "✓ Hybrid repeat/discovery candidates evaluated."
)

print(
    "✓ Performance holdout remains unused."
)

## Step 59 Summary — Item Affinity and Hybrid Recommendation

Step 59 introduced item-to-item product affinity and combined it with customer repeat-frequency recommendations.

### Item Affinity Improved Cross-Sell Personalisation

Among episodes containing at least one new-to-customer product, item affinity substantially outperformed popularity-based cross-sell.

At Top-10:

- **Popularity Cross-Sell Hit Rate: 15.6%**
- **Affinity Cross-Sell Hit Rate: 25.4%**

Recall improved from approximately **2.0% to 4.4%**.

Precision improved from approximately **2.0% to 4.0%**.

MRR improved from approximately **0.065 to 0.111**.

Item affinity therefore provides genuine incremental value for product discovery.

### Repeat Frequency Remained the Strongest General Recommender

For the full next-basket problem, customer repeat frequency remained substantially stronger than item affinity.

At Top-10:

- **Repeat Frequency Hit Rate: 70.6%**
- **Item Affinity Hit Rate: 53.2%**
- **Global Popularity Hit Rate: 43.4%**

The recommendation architecture should therefore preserve repeat-frequency ranking as the primary replenishment channel.

### Hybrid Weighting Introduced an Accuracy–Discovery Trade-Off

At Top-20:

- **Repeat Frequency Hit Rate: 79.2%**
- **Hybrid 70R / 30D: 78.0%**
- **Hybrid 60R / 40D: 76.5%**
- **Hybrid 50R / 50D: 74.9%**

As more recommendation slots are allocated to discovery, total next-basket accuracy declines.

This represents the expected trade-off between immediate predictive accuracy and product exploration.

### 70R / 30D Was the Strongest Hybrid Candidate

Among tested hybrid allocations, 70% repeat and 30% discovery produced the strongest overall performance.

At Top-20 it achieved approximately:

- **Hit Rate: 78.0%**
- **Recall: 23.7%**
- **Precision: 13.9%**
- **MRR: 0.419**
- **Catalogue Coverage: 46.1%**

This retains most of the predictive strength of repeat-frequency ranking while introducing a dedicated discovery channel.

## Architecture Position

The preferred recommendation design is:

### Repeat / Replenishment Channel
Customer Repeat Frequency

### Discovery / Cross-Sell Channel
Item-to-Item Affinity

### Combined Candidate
70% repeat / 30% discovery

This candidate should now undergo temporal robustness testing before the final recommendation architecture is frozen.

## Step 60 — Temporal Robustness of the Recommendation Architecture

Step 60 evaluates the leading recommendation components across historical out-of-time periods.

The candidate architecture is:

- **Repeat Channel:** Customer Repeat Frequency
- **Discovery Channel:** Item-to-Item Affinity
- **Hybrid Allocation:** 70% Repeat / 30% Discovery

Historical validation periods:

- September 2010
- October 2010
- November 2010

For each month:

- global product statistics are built only from baskets available before the month begins;
- customer-specific histories use only information available up to the prediction order;
- no later recommendation outcomes are used to construct the recommender.

The Jul–Aug 2011 performance holdout remains unused.

In [ ]:
# ================================================================
# 60.2 — HISTORICAL RECOMMENDATION FOLDS
# ================================================================

REC_PURGED_VALIDATION_PERIODS = [
    pd.Period(
        "2010-09",
        freq="M"
    ),

    pd.Period(
        "2010-10",
        freq="M"
    ),

    pd.Period(
        "2010-11",
        freq="M"
    ),
]


print(
    REC_PURGED_VALIDATION_PERIODS
)

In [ ]:
# ================================================================
# 60.3 — HISTORICAL MONTH-START SNAPSHOTS
# ================================================================

historical_affinity_snapshots = {}
historical_popularity_snapshots = {}


for period in (
    REC_PURGED_VALIDATION_PERIODS
):

    cutoff = (
        period.to_timestamp(
            how="start"
        )
    )


    past_baskets = (
        historical_baskets.loc[
            historical_baskets[
                "order_date"
            ]
            <
            cutoff
        ]
        .copy()
    )


    affinity, item_count = (
        build_item_affinity(
            baskets=
                past_baskets[
                    "products"
                ],

            minimum_item_orders=5,

            minimum_pair_orders=2
        )
    )


    historical_affinity_snapshots[
        period
    ] = affinity


    historical_popularity_snapshots[
        period
    ] = [
        item
        for item, count
        in sorted(
            item_count.items(),
            key=lambda x:
                x[1],
            reverse=True
        )
    ]


    print(
        period,
        "baskets:",
        len(
            past_baskets
        ),
        "affinity sources:",
        len(
            affinity
        )
    )

In [ ]:
# ================================================================
# 60.4 — HISTORICAL FOLD EPISODES
# ================================================================

recommendation_temporal_folds = {}


for period in (
    REC_PURGED_VALIDATION_PERIODS
):

    fold_episodes = (
        next_basket_episodes.loc[
            next_basket_episodes[
                "prediction_period"
            ]
            ==
            period
        ]
        .copy()
        .reset_index(drop=True)
    )


    recommendation_temporal_folds[
        period
    ] = fold_episodes


    print(
        period,
        "episodes:",
        len(
            fold_episodes
        )
    )

In [ ]:
# ================================================================
# 60.5 — GENERATE FOLD RECOMMENDATIONS
# ================================================================

def generate_recommendations_for_fold(
    episodes,
    period,
    top_n=20
):

    repeat_lists = []
    discovery_lists = []
    hybrid_lists = []


    affinity_dictionary = (
        historical_affinity_snapshots[
            period
        ]
    )


    popularity_list = (
        historical_popularity_snapshots[
            period
        ]
    )


    for row in (
        episodes.itertuples(
            index=False
        )
    ):

        history_summary = (
            customer_product_summary_at_date(
                customer=
                    row.customer,

                cutoff=
                    row.order_date,

                history_table=
                    customer_product_orders
            )
        )


        # Repeat channel
        repeat_history = (
            customer_product_orders.loc[
                (
                    customer_product_orders[
                        "customer"
                    ]
                    ==
                    row.customer
                )
                &
                (
                    customer_product_orders[
                        "date"
                    ]
                    <=
                    row.order_date
                )
            ]
            .copy()
        )


        if repeat_history.empty:

            repeat_ranked = []

        else:

            repeat_ranked = (
                repeat_history
                .groupby(
                    "product",
                    as_index=False
                )
                .agg(
                    purchase_orders=(
                        "invoice",
                        "nunique"
                    ),

                    last_purchase=(
                        "date",
                        "max"
                    ),
                )
                .sort_values(
                    [
                        "purchase_orders",
                        "last_purchase",
                    ],
                    ascending=[
                        False,
                        False,
                    ]
                )[
                    "product"
                ]
                .head(
                    top_n
                )
                .tolist()
            )


        # Discovery channel
        discovery_ranked = (
            rank_affinity_recommendations(
                product_history_summary=
                    history_summary,

                affinity_dictionary=
                    affinity_dictionary,

                popularity_list=
                    popularity_list,

                top_n=
                    top_n,

                exclude_history=
                    True
            )
        )


        # Hybrid 70/30
        hybrid_ranked = (
            build_hybrid_list(
                repeat_list=
                    repeat_ranked,

                discovery_list=
                    discovery_ranked,

                top_n=
                    top_n,

                repeat_fraction=
                    0.70
            )
        )


        repeat_lists.append(
            repeat_ranked
        )

        discovery_lists.append(
            discovery_ranked
        )

        hybrid_lists.append(
            hybrid_ranked
        )


    return (
        repeat_lists,
        discovery_lists,
        hybrid_lists,
    )

In [ ]:
# ================================================================
# 60.6 — HISTORICAL TEMPORAL EVALUATION
# ================================================================

recommendation_temporal_results = []


for period, episodes in (
    recommendation_temporal_folds.items()
):

    (
        repeat_lists,
        discovery_lists,
        hybrid_lists,
    ) = (
        generate_recommendations_for_fold(
            episodes=
                episodes,

            period=
                period,

            top_n=
                20
        )
    )


    # ------------------------------------------------------------
    # REPEAT — GENERAL NEXT BASKET
    # ------------------------------------------------------------

    repeat_results = (
        evaluate_recommendation_lists(
            actual_baskets=
                episodes[
                    "next_basket_products"
                ],

            recommendation_lists=
                repeat_lists,

            model_name=
                "Repeat Frequency",

            ks=(
                5,
                10,
                20,
            )
        )
    )


    repeat_results[
        "Validation Period"
    ] = str(
        period
    )


    repeat_results[
        "Evaluation"
    ] = (
        "General Next Basket"
    )


    recommendation_temporal_results.append(
        repeat_results
    )


    # ------------------------------------------------------------
    # HYBRID — GENERAL NEXT BASKET
    # ------------------------------------------------------------

    hybrid_results = (
        evaluate_recommendation_lists(
            actual_baskets=
                episodes[
                    "next_basket_products"
                ],

            recommendation_lists=
                hybrid_lists,

            model_name=
                "Hybrid 70R / 30D",

            ks=(
                5,
                10,
                20,
            )
        )
    )


    hybrid_results[
        "Validation Period"
    ] = str(
        period
    )


    hybrid_results[
        "Evaluation"
    ] = (
        "General Next Basket"
    )


    recommendation_temporal_results.append(
        hybrid_results
    )


    # ------------------------------------------------------------
    # DISCOVERY — ELIGIBLE CROSS-SELL
    # ------------------------------------------------------------

    eligible_mask = (
        episodes[
            "new_product_count"
        ]
        >
        0
    )


    eligible_episodes = (
        episodes.loc[
            eligible_mask
        ]
    )


    eligible_discovery_lists = [
        recommendations
        for recommendations, eligible
        in zip(
            discovery_lists,
            eligible_mask
        )
        if eligible
    ]


    discovery_results = (
        evaluate_recommendation_lists(
            actual_baskets=
                eligible_episodes[
                    "new_products"
                ],

            recommendation_lists=
                eligible_discovery_lists,

            model_name=
                "Affinity Discovery",

            ks=(
                5,
                10,
                20,
            )
        )
    )


    discovery_results[
        "Validation Period"
    ] = str(
        period
    )


    discovery_results[
        "Evaluation"
    ] = (
        "Cross-Sell Eligible"
    )


    recommendation_temporal_results.append(
        discovery_results
    )

In [ ]:
recommendation_temporal_results = (
    pd.concat(
        recommendation_temporal_results,
        ignore_index=True
    )
)


display(
    recommendation_temporal_results
)

In [ ]:
# ================================================================
# 60.7 — TEMPORAL SUMMARY
# ================================================================

recommendation_temporal_summary = (
    recommendation_temporal_results
    .groupby(
        [
            "Model",
            "Evaluation",
            "K",
        ],
        as_index=False
    )
    .agg(
        folds=(
            "Validation Period",
            "nunique"
        ),

        mean_Hit_Rate=(
            "Hit Rate",
            "mean"
        ),

        min_Hit_Rate=(
            "Hit Rate",
            "min"
        ),

        mean_Recall=(
            "Recall",
            "mean"
        ),

        min_Recall=(
            "Recall",
            "min"
        ),

        mean_Precision=(
            "Precision",
            "mean"
        ),

        mean_MRR=(
            "MRR",
            "mean"
        ),
    )
)


display(
    recommendation_temporal_summary
)

In [ ]:
# ================================================================
# 60.8 — HYBRID VS REPEAT TOP-20
# ================================================================

top20_general = (
    recommendation_temporal_results.loc[
        (
            recommendation_temporal_results[
                "Evaluation"
            ]
            ==
            "General Next Basket"
        )
        &
        (
            recommendation_temporal_results[
                "K"
            ]
            ==
            20
        )
    ]
    .copy()
)


top20_general_pivot = (
    top20_general
    .pivot(
        index=
            "Validation Period",

        columns=
            "Model",

        values=[
            "Hit Rate",
            "Recall",
            "Precision",
            "MRR",
        ]
    )
)


display(
    top20_general_pivot
)

In [ ]:
# ================================================================
# 60.9 — HYBRID ACCURACY COST
# ================================================================

hybrid_tradeoff_records = []


for period in (
    REC_PURGED_VALIDATION_PERIODS
):

    period_str = str(
        period
    )


    period_results = (
        top20_general.loc[
            top20_general[
                "Validation Period"
            ]
            ==
            period_str
        ]
    )


    repeat_row = (
        period_results.loc[
            period_results[
                "Model"
            ]
            ==
            "Repeat Frequency"
        ]
        .iloc[0]
    )


    hybrid_row = (
        period_results.loc[
            period_results[
                "Model"
            ]
            ==
            "Hybrid 70R / 30D"
        ]
        .iloc[0]
    )


    hybrid_tradeoff_records.append(
        {
            "Validation Period":
                period_str,

            "Repeat Hit Rate":
                repeat_row[
                    "Hit Rate"
                ],

            "Hybrid Hit Rate":
                hybrid_row[
                    "Hit Rate"
                ],

            "Hit Rate Difference":
                (
                    hybrid_row[
                        "Hit Rate"
                    ]
                    -
                    repeat_row[
                        "Hit Rate"
                    ]
                ),

            "Repeat Recall":
                repeat_row[
                    "Recall"
                ],

            "Hybrid Recall":
                hybrid_row[
                    "Recall"
                ],

            "Recall Difference":
                (
                    hybrid_row[
                        "Recall"
                    ]
                    -
                    repeat_row[
                        "Recall"
                    ]
                ),
        }
    )


hybrid_temporal_tradeoff = pd.DataFrame(
    hybrid_tradeoff_records
)


display(
    hybrid_temporal_tradeoff
)

In [ ]:
# ================================================================
# 60.10 — DISCOVERY STABILITY
# ================================================================

discovery_temporal_stability = (
    recommendation_temporal_results.loc[
        (
            recommendation_temporal_results[
                "Model"
            ]
            ==
            "Affinity Discovery"
        )
        &
        (
            recommendation_temporal_results[
                "K"
            ]
            ==
            10
        ),
        [
            "Validation Period",
            "Hit Rate",
            "Recall",
            "Precision",
            "MRR",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    discovery_temporal_stability
)

In [ ]:
# ================================================================
# 60.11 — PRE-HOLDOUT SCORECARD
# ================================================================

historical_repeat_20 = (
    recommendation_temporal_summary.loc[
        (
            recommendation_temporal_summary[
                "Model"
            ]
            ==
            "Repeat Frequency"
        )
        &
        (
            recommendation_temporal_summary[
                "K"
            ]
            ==
            20
        )
    ]
    .iloc[0]
)


historical_hybrid_20 = (
    recommendation_temporal_summary.loc[
        (
            recommendation_temporal_summary[
                "Model"
            ]
            ==
            "Hybrid 70R / 30D"
        )
        &
        (
            recommendation_temporal_summary[
                "K"
            ]
            ==
            20
        )
    ]
    .iloc[0]
)


historical_discovery_10 = (
    recommendation_temporal_summary.loc[
        (
            recommendation_temporal_summary[
                "Model"
            ]
            ==
            "Affinity Discovery"
        )
        &
        (
            recommendation_temporal_summary[
                "K"
            ]
            ==
            10
        )
    ]
    .iloc[0]
)


recommendation_preholdout_scorecard = (
    pd.DataFrame(
        [
            {
                "Component":
                    "Repeat Frequency",

                "Metric":
                    "Hit Rate@20",

                "Designated Validation":
                    repeat_baseline_results.loc[
                        repeat_baseline_results[
                            "K"
                        ]
                        ==
                        20,
                        "Hit Rate"
                    ].iloc[0],

                "Historical Mean":
                    historical_repeat_20[
                        "mean_Hit_Rate"
                    ],

                "Historical Minimum":
                    historical_repeat_20[
                        "min_Hit_Rate"
                    ],
            },

            {
                "Component":
                    "Hybrid 70R / 30D",

                "Metric":
                    "Hit Rate@20",

                "Designated Validation":
                    hybrid_recommendation_results.loc[
                        (
                            hybrid_recommendation_results[
                                "Model"
                            ]
                            ==
                            "Hybrid 70R / 30D"
                        )
                        &
                        (
                            hybrid_recommendation_results[
                                "K"
                            ]
                            ==
                            20
                        ),
                        "Hit Rate"
                    ].iloc[0],

                "Historical Mean":
                    historical_hybrid_20[
                        "mean_Hit_Rate"
                    ],

                "Historical Minimum":
                    historical_hybrid_20[
                        "min_Hit_Rate"
                    ],
            },

            {
                "Component":
                    "Affinity Discovery",

                "Metric":
                    "Cross-Sell Hit Rate@10",

                "Designated Validation":
                    affinity_cross_sell_eligible_results.loc[
                        affinity_cross_sell_eligible_results[
                            "K"
                        ]
                        ==
                        10,
                        "Hit Rate"
                    ].iloc[0],

                "Historical Mean":
                    historical_discovery_10[
                        "mean_Hit_Rate"
                    ],

                "Historical Minimum":
                    historical_discovery_10[
                        "min_Hit_Rate"
                    ],
            },
        ]
    )
)


display(
    recommendation_preholdout_scorecard
)

In [ ]:
# ================================================================
# 60.12 — DECISION PENDING
# ================================================================

RECOMMENDATION_ARCHITECTURE_FROZEN = False


print(
    "Recommendation architecture remains unfrozen "
    "pending temporal robustness review."
)

In [ ]:
# ================================================================
# 60.13 — GOVERNANCE
# ================================================================

assert (
    RECOMMENDATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Jul–Aug recommendation performance holdout remains unused."
)

In [ ]:
# ================================================================
# 60.14 — STEP 60 COMPLETE
# ================================================================

assert len(
    recommendation_temporal_results
) > 0


assert (
    RECOMMENDATION_TEST_SET_OPENED
    is False
)


print(
    "✓ Step 60 complete."
)

print(
    "✓ Repeat channel tested historically."
)

print(
    "✓ Discovery affinity tested historically."
)

print(
    "✓ Hybrid 70R / 30D tested historically."
)

print(
    "✓ Jul–Aug performance holdout remains unused."
)

## Step 60 Summary — Temporal Robustness and Architecture Freeze

Step 60 evaluated the leading recommendation components across three historical out-of-time periods.

The analysis compared:

- Customer Repeat Frequency;
- Item-Affinity Discovery;
- and a 70% Repeat / 30% Discovery combined list.

### Repeat Frequency Was Temporally Robust

At Top-20, Customer Repeat Frequency achieved historical Hit Rates of approximately:

- **74.4% — September 2010**
- **76.4% — October 2010**
- **77.3% — November 2010**

Historical mean Hit Rate@20 was approximately:

> **76.0%**

The designated Validation Hit Rate@20 was approximately:

> **79.2%**

The replenishment signal therefore demonstrated strong and stable out-of-time performance.

### Item Affinity Discovery Was Highly Stable

Cross-sell Hit Rate@10 was approximately:

- **26.2% — September 2010**
- **24.8% — October 2010**
- **26.4% — November 2010**

Historical mean was approximately:

> **25.8%**

The designated Validation result was approximately:

> **25.4%**

Item affinity therefore demonstrated strong temporal consistency as a new-product discovery mechanism.

### The Combined Hybrid Introduced a Consistent Accuracy Cost

The 70% Repeat / 30% Discovery list produced lower Top-20 general recommendation accuracy than pure Repeat Frequency in every historical period.

The Hit Rate cost ranged from approximately:

> **1.3 to 2.7 percentage points**

Recall was also consistently lower.

This does not imply that discovery recommendations lack value.

Instead, it demonstrates that repeat and discovery represent distinct recommendation objectives.

### Final Pre-Holdout Architecture

The recommendation engine is therefore frozen as two separate channels.

#### Replenishment Channel

> **Customer Repeat Frequency**

#### Discovery Channel

> **Item-to-Item Affinity**

The previously tested 70% Repeat / 30% Discovery allocation may be retained as an optional presentation layer when one combined list is operationally required, but it is not the primary predictive architecture.

The Jul–Aug 2011 recommendation performance holdout remains unevaluated.


## STEP 61 — FREEZE ARCHITECTURE AND OPEN PERFORMANCE HOLDOUT
We now freeze everything.

No changes after seeing July–August recommendation performance.

In [ ]:
# ================================================================
# 61.1 — FREEZE RECOMMENDATION ARCHITECTURE
# ================================================================

FINAL_REPEAT_RECOMMENDER = (
    "Customer Repeat Frequency"
)

FINAL_DISCOVERY_RECOMMENDER = (
    "Item-to-Item Affinity"
)

FINAL_OPTIONAL_HYBRID = (
    "70% Repeat / 30% Discovery"
)


RECOMMENDATION_REPEAT_CHANNEL_FROZEN = True
RECOMMENDATION_DISCOVERY_CHANNEL_FROZEN = True
RECOMMENDATION_ARCHITECTURE_FROZEN = True


print(
    f"Repeat channel: "
    f"{FINAL_REPEAT_RECOMMENDER}"
)

print(
    f"Discovery channel: "
    f"{FINAL_DISCOVERY_RECOMMENDER}"
)

print(
    f"Optional combined presentation: "
    f"{FINAL_OPTIONAL_HYBRID}"
)

print(
    "✓ Recommendation architecture frozen."
)

In [ ]:
# ================================================================
# 61.2 — FROZEN SPECIFICATION
# ================================================================

recommendation_frozen_specification = pd.DataFrame(
    {
        "Component": [
            "Prediction target",
            "Outcome horizon",
            "Repeat channel",
            "Repeat ranking",
            "Discovery channel",
            "Discovery similarity",
            "Minimum item orders",
            "Minimum pair orders",
            "Primary general metric",
            "Primary discovery metric",
            "Evaluation depths",
            "Optional combined display",
            "Holdout status",
        ],

        "Frozen Specification": [
            "Products in next qualifying order",
            "Next order within 90 days",
            "Customer Repeat Frequency",
            "Purchase frequency, then recency tie-break",
            "Item-to-Item Affinity",
            "Cosine-style basket co-occurrence",
            5,
            2,
            "Hit Rate / Recall",
            "Cross-Sell Hit Rate",
            "Top-5, Top-10, Top-20",
            "70% Repeat / 30% Discovery",
            "NOT YET EVALUATED",
        ],
    }
)


display(
    recommendation_frozen_specification
)

In [ ]:
# ================================================================
# 61.3 — HOLDOUT MONTHS
# ================================================================

recommendation_holdout_periods = [
    pd.Period(
        "2011-07",
        freq="M"
    ),

    pd.Period(
        "2011-08",
        freq="M"
    ),
]


print(
    recommendation_holdout_periods
)

### 61.4 — Build holdout month-start affinity snapshots
This is important.

July recommendations use only baskets occurring before 1 July.

August recommendations use only baskets occurring before 1 August.

In [ ]:
# ================================================================
# 61.4 — HOLDOUT AFFINITY SNAPSHOTS
# ================================================================

holdout_affinity_snapshots = {}
holdout_popularity_snapshots = {}


for period in (
    recommendation_holdout_periods
):

    cutoff = (
        period.to_timestamp(
            how="start"
        )
    )


    past_baskets = (
        historical_baskets.loc[
            historical_baskets[
                "order_date"
            ]
            <
            cutoff
        ]
        .copy()
    )


    affinity, item_count = (
        build_item_affinity(
            baskets=
                past_baskets[
                    "products"
                ],

            minimum_item_orders=5,

            minimum_pair_orders=2
        )
    )


    holdout_affinity_snapshots[
        period
    ] = affinity


    holdout_popularity_snapshots[
        period
    ] = [
        product
        for product, count
        in sorted(
            item_count.items(),
            key=lambda item:
                item[1],
            reverse=True
        )
    ]


    print(
        period,
        "historical baskets:",
        len(
            past_baskets
        ),
        "affinity source products:",
        len(
            affinity
        )
    )

In [ ]:
# ================================================================
# 61.5 — HOLDOUT RECOMMENDATION GENERATOR
# ================================================================

def generate_holdout_recommendations(
    episodes,
    period,
    top_n=20
):

    repeat_lists = []
    discovery_lists = []
    hybrid_lists = []


    affinity_dictionary = (
        holdout_affinity_snapshots[
            period
        ]
    )


    popularity_list = (
        holdout_popularity_snapshots[
            period
        ]
    )


    for row in (
        episodes.itertuples(
            index=False
        )
    ):

        # --------------------------------------------------------
        # Customer history available at prediction
        # --------------------------------------------------------

        history_summary = (
            customer_product_summary_at_date(
                customer=
                    row.customer,

                cutoff=
                    row.order_date,

                history_table=
                    customer_product_orders
            )
        )


        repeat_history = (
            customer_product_orders.loc[
                (
                    customer_product_orders[
                        "customer"
                    ]
                    ==
                    row.customer
                )
                &
                (
                    customer_product_orders[
                        "date"
                    ]
                    <=
                    row.order_date
                )
            ]
            .copy()
        )


        # --------------------------------------------------------
        # REPEAT CHANNEL
        # --------------------------------------------------------

        if repeat_history.empty:

            repeat_ranked = []

        else:

            repeat_ranked = (
                repeat_history
                .groupby(
                    "product",
                    as_index=False
                )
                .agg(
                    purchase_orders=(
                        "invoice",
                        "nunique"
                    ),

                    last_purchase=(
                        "date",
                        "max"
                    ),
                )
                .sort_values(
                    [
                        "purchase_orders",
                        "last_purchase",
                    ],
                    ascending=[
                        False,
                        False,
                    ]
                )[
                    "product"
                ]
                .head(
                    top_n
                )
                .tolist()
            )


        # --------------------------------------------------------
        # DISCOVERY CHANNEL
        # --------------------------------------------------------

        discovery_ranked = (
            rank_affinity_recommendations(
                product_history_summary=
                    history_summary,

                affinity_dictionary=
                    affinity_dictionary,

                popularity_list=
                    popularity_list,

                top_n=
                    top_n,

                exclude_history=
                    True
            )
        )


        # --------------------------------------------------------
        # OPTIONAL HYBRID
        # --------------------------------------------------------

        hybrid_ranked = (
            build_hybrid_list(
                repeat_list=
                    repeat_ranked,

                discovery_list=
                    discovery_ranked,

                top_n=
                    top_n,

                repeat_fraction=
                    0.70
            )
        )


        repeat_lists.append(
            repeat_ranked
        )

        discovery_lists.append(
            discovery_ranked
        )

        hybrid_lists.append(
            hybrid_ranked
        )


    return (
        repeat_lists,
        discovery_lists,
        hybrid_lists,
    )

In [ ]:
# ================================================================
# 61.6 — GENERATE HOLDOUT RECOMMENDATIONS
# ================================================================

holdout_prediction_frames = []


for period in (
    recommendation_holdout_periods
):

    episodes = (
        recommendation_test.loc[
            recommendation_test[
                "prediction_period"
            ]
            ==
            period
        ]
        .copy()
        .reset_index(drop=True)
    )


    (
        repeat_lists,
        discovery_lists,
        hybrid_lists,
    ) = (
        generate_holdout_recommendations(
            episodes=
                episodes,

            period=
                period,

            top_n=
                20
        )
    )


    fold = (
        episodes.copy()
    )


    fold[
        "repeat_recommendations"
    ] = repeat_lists


    fold[
        "discovery_recommendations"
    ] = discovery_lists


    fold[
        "hybrid_recommendations"
    ] = hybrid_lists


    holdout_prediction_frames.append(
        fold
    )


recommendation_holdout_predictions = (
    pd.concat(
        holdout_prediction_frames,
        ignore_index=True
    )
)


print(
    f"Holdout prediction episodes: "
    f"{len(recommendation_holdout_predictions):,}"
)

In [ ]:
# ================================================================
# 61.7 — OPEN PERFORMANCE HOLDOUT
# ================================================================

RECOMMENDATION_TEST_SET_OPENED = True


print(
    "RECOMMENDATION PERFORMANCE HOLDOUT IS NOW OPEN."
)

print(
    "No architecture changes are permitted after this point."
)

In [ ]:
# ================================================================
# 61.8 — HOLDOUT TARGET SUMMARY
# ================================================================

recommendation_holdout_summary = pd.DataFrame(
    {
        "Metric": [
            "Episodes",
            "Customers",
            "Median next-basket size",
            "Average next-basket size",
            "Average repeat share",
            "Average new-product share",
            "Cross-sell eligible rate",
        ],

        "Value": [
            len(
                recommendation_holdout_predictions
            ),

            recommendation_holdout_predictions[
                "customer"
            ].nunique(),

            recommendation_holdout_predictions[
                "next_basket_size"
            ].median(),

            recommendation_holdout_predictions[
                "next_basket_size"
            ].mean(),

            recommendation_holdout_predictions[
                "repeat_share"
            ].mean(),

            recommendation_holdout_predictions[
                "new_product_share"
            ].mean(),

            (
                recommendation_holdout_predictions[
                    "new_product_count"
                ]
                >
                0
            ).mean(),
        ],
    }
)


display(
    recommendation_holdout_summary
)

In [ ]:
# ================================================================
# 61.9 — REPEAT CHANNEL HOLDOUT RESULTS
# ================================================================

final_repeat_channel_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_holdout_predictions[
                "next_basket_products"
            ],

        recommendation_lists=
            recommendation_holdout_predictions[
                "repeat_recommendations"
            ],

        model_name=
            "Frozen Repeat Frequency — Holdout",

        ks=(
            5,
            10,
            20,
        )
    )
)


display(
    final_repeat_channel_results
)

In [ ]:
# ================================================================
# 61.10 — REPEAT-ELIGIBLE HOLDOUT RESULTS
# ================================================================

holdout_repeat_mask = (
    recommendation_holdout_predictions[
        "repeat_product_count"
    ]
    >
    0
)


holdout_repeat_eligible = (
    recommendation_holdout_predictions.loc[
        holdout_repeat_mask
    ]
    .copy()
)


holdout_repeat_lists = (
    recommendation_holdout_predictions.loc[
        holdout_repeat_mask,
        "repeat_recommendations"
    ]
    .tolist()
)


final_repeat_eligible_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            holdout_repeat_eligible[
                "repeat_products"
            ],

        recommendation_lists=
            holdout_repeat_lists,

        model_name=
            "Frozen Repeat Frequency — Repeat Eligible",

        ks=(
            5,
            10,
            20,
        )
    )
)


display(
    final_repeat_eligible_results
)

In [ ]:
# ================================================================
# 61.11 — DISCOVERY CHANNEL HOLDOUT RESULTS
# ================================================================

holdout_discovery_mask = (
    recommendation_holdout_predictions[
        "new_product_count"
    ]
    >
    0
)


holdout_discovery_eligible = (
    recommendation_holdout_predictions.loc[
        holdout_discovery_mask
    ]
    .copy()
)


holdout_discovery_lists = (
    recommendation_holdout_predictions.loc[
        holdout_discovery_mask,
        "discovery_recommendations"
    ]
    .tolist()
)


final_discovery_channel_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            holdout_discovery_eligible[
                "new_products"
            ],

        recommendation_lists=
            holdout_discovery_lists,

        model_name=
            "Frozen Affinity Discovery — Holdout",

        ks=(
            5,
            10,
            20,
        )
    )
)


display(
    final_discovery_channel_results
)

In [ ]:
# ================================================================
# 61.12 — OPTIONAL HYBRID HOLDOUT RESULTS
# ================================================================

final_hybrid_results = (
    evaluate_recommendation_lists(
        actual_baskets=
            recommendation_holdout_predictions[
                "next_basket_products"
            ],

        recommendation_lists=
            recommendation_holdout_predictions[
                "hybrid_recommendations"
            ],

        model_name=
            "Frozen 70R / 30D Presentation — Holdout",

        ks=(
            5,
            10,
            20,
        )
    )
)


display(
    final_hybrid_results
)

In [ ]:
# ================================================================
# 61.13 — PREVIOUS EVIDENCE VS HOLDOUT
# ================================================================

recommendation_validation_vs_holdout = pd.DataFrame(
    [
        {
            "Component":
                "Repeat Frequency",

            "Metric":
                "Hit Rate@20",

            "Historical Mean":
                historical_repeat_20[
                    "mean_Hit_Rate"
                ],

            "Designated Validation":
                repeat_baseline_results.loc[
                    repeat_baseline_results[
                        "K"
                    ]
                    ==
                    20,
                    "Hit Rate"
                ].iloc[0],

            "Holdout":
                final_repeat_channel_results.loc[
                    final_repeat_channel_results[
                        "K"
                    ]
                    ==
                    20,
                    "Hit Rate"
                ].iloc[0],
        },

        {
            "Component":
                "Affinity Discovery",

            "Metric":
                "Cross-Sell Hit Rate@10",

            "Historical Mean":
                historical_discovery_10[
                    "mean_Hit_Rate"
                ],

            "Designated Validation":
                affinity_cross_sell_eligible_results.loc[
                    affinity_cross_sell_eligible_results[
                        "K"
                    ]
                    ==
                    10,
                    "Hit Rate"
                ].iloc[0],

            "Holdout":
                final_discovery_channel_results.loc[
                    final_discovery_channel_results[
                        "K"
                    ]
                    ==
                    10,
                    "Hit Rate"
                ].iloc[0],
        },

        {
            "Component":
                "Optional Hybrid 70/30",

            "Metric":
                "Hit Rate@20",

            "Historical Mean":
                historical_hybrid_20[
                    "mean_Hit_Rate"
                ],

            "Designated Validation":
                hybrid_recommendation_results.loc[
                    (
                        hybrid_recommendation_results[
                            "Model"
                        ]
                        ==
                        "Hybrid 70R / 30D"
                    )
                    &
                    (
                        hybrid_recommendation_results[
                            "K"
                        ]
                        ==
                        20
                    ),
                    "Hit Rate"
                ].iloc[0],

            "Holdout":
                final_hybrid_results.loc[
                    final_hybrid_results[
                        "K"
                    ]
                    ==
                    20,
                    "Hit Rate"
                ].iloc[0],
        },
    ]
)


recommendation_validation_vs_holdout[
    "Holdout vs Historical Mean"
] = (
    recommendation_validation_vs_holdout[
        "Holdout"
    ]
    -
    recommendation_validation_vs_holdout[
        "Historical Mean"
    ]
)


recommendation_validation_vs_holdout[
    "Holdout vs Designated Validation"
] = (
    recommendation_validation_vs_holdout[
        "Holdout"
    ]
    -
    recommendation_validation_vs_holdout[
        "Designated Validation"
    ]
)


display(
    recommendation_validation_vs_holdout
)

In [ ]:
# ================================================================
# 61.14 — FINAL CATALOGUE COVERAGE
# ================================================================

holdout_end_date = (
    recommendation_holdout_predictions[
        "order_date"
    ]
    .max()
)


holdout_available_catalogue = (
    customer_product_orders.loc[
        customer_product_orders[
            "date"
        ]
        <=
        holdout_end_date,
        "product"
    ]
    .unique()
)


final_recommendation_coverage_records = []


for (
    model_name,
    recommendation_column
) in [
    (
        "Repeat Frequency",
        "repeat_recommendations"
    ),
    (
        "Affinity Discovery",
        "discovery_recommendations"
    ),
    (
        "Optional Hybrid 70/30",
        "hybrid_recommendations"
    ),
]:

    for k in [
        5,
        10,
        20,
    ]:

        coverage = (
            recommendation_catalogue_coverage(
                recommendation_lists=
                    recommendation_holdout_predictions[
                        recommendation_column
                    ],

                catalogue_products=
                    holdout_available_catalogue,

                k=
                    k
            )
        )


        final_recommendation_coverage_records.append(
            {
                "Model":
                    model_name,

                "K":
                    k,

                "Catalogue Coverage":
                    coverage,

                "Catalogue Coverage (%)":
                    coverage
                    *
                    100,
            }
        )


final_recommendation_catalogue_coverage = (
    pd.DataFrame(
        final_recommendation_coverage_records
    )
)


display(
    final_recommendation_catalogue_coverage
)

In [ ]:
# ================================================================
# 61.15 — HOLDOUT MONTH STABILITY
# ================================================================

holdout_monthly_results = []


for period in (
    recommendation_holdout_periods
):

    month_data = (
        recommendation_holdout_predictions.loc[
            recommendation_holdout_predictions[
                "prediction_period"
            ]
            ==
            period
        ]
        .copy()
    )


    # Repeat @20
    repeat_result = (
        evaluate_recommendation_lists(
            actual_baskets=
                month_data[
                    "next_basket_products"
                ],

            recommendation_lists=
                month_data[
                    "repeat_recommendations"
                ],

            model_name=
                "Repeat Frequency",

            ks=(
                20,
            )
        )
    )


    repeat_result[
        "Period"
    ] = str(
        period
    )


    repeat_result[
        "Evaluation"
    ] = (
        "General"
    )


    holdout_monthly_results.append(
        repeat_result
    )


    # Discovery @10
    discovery_mask = (
        month_data[
            "new_product_count"
        ]
        >
        0
    )


    discovery_data = (
        month_data.loc[
            discovery_mask
        ]
    )


    discovery_result = (
        evaluate_recommendation_lists(
            actual_baskets=
                discovery_data[
                    "new_products"
                ],

            recommendation_lists=
                discovery_data[
                    "discovery_recommendations"
                ],

            model_name=
                "Affinity Discovery",

            ks=(
                10,
            )
        )
    )


    discovery_result[
        "Period"
    ] = str(
        period
    )


    discovery_result[
        "Evaluation"
    ] = (
        "Cross-Sell Eligible"
    )


    holdout_monthly_results.append(
        discovery_result
    )


holdout_monthly_results = (
    pd.concat(
        holdout_monthly_results,
        ignore_index=True
    )
)


display(
    holdout_monthly_results
)

In [ ]:
# ================================================================
# 61.16 — HOLDOUT CONSUMED
# ================================================================

RECOMMENDATION_TEST_SET_CONSUMED = True


assert (
    RECOMMENDATION_ARCHITECTURE_FROZEN
    is True
)

assert (
    RECOMMENDATION_TEST_SET_OPENED
    is True
)

assert (
    RECOMMENDATION_TEST_SET_CONSUMED
    is True
)


print(
    "✓ Recommendation performance holdout consumed."
)

print(
    "✓ No post-holdout architecture changes permitted."
)

## Step 61 Summary — Final Recommendation Holdout

The frozen recommendation architecture was evaluated on the July–August 2011 performance holdout.

The architecture contained two distinct recommendation channels:

### Replenishment
Customer Repeat Frequency

### Discovery
Item-to-Item Affinity

An optional 70% Repeat / 30% Discovery presentation layer was also retained.

### Repeat Recommendation Generalised Extremely Well

The repeat-frequency channel achieved approximately:

- **Top-5 Hit Rate: 59.6%**
- **Top-10 Hit Rate: 70.6%**
- **Top-20 Hit Rate: 79.2%**

Among episodes containing repeat products, Top-20 Hit Rate reached approximately:

> **87.1%**

Holdout Top-20 Hit Rate was effectively unchanged from the designated Validation result:

- **Validation: 79.19%**
- **Holdout: 79.23%**

The replenishment recommender therefore demonstrated excellent out-of-time stability.

### Discovery Affinity Remained Useful but Weakened

The affinity discovery channel achieved approximately:

- **Top-5 Hit Rate: 15.4%**
- **Top-10 Hit Rate: 21.6%**
- **Top-20 Hit Rate: 29.9%**

Top-10 discovery Hit Rate declined from approximately **25.4% on Validation** to **21.6% on Holdout**.

The decline was visible in both July and August and therefore appears to represent genuine temporal variation rather than one isolated month.

Affinity discovery remains useful as a personalized cross-sell mechanism but is less stable than repeat recommendation.

### Optional Hybrid Remained Stable

The 70% Repeat / 30% Discovery presentation achieved a Holdout Top-20 Hit Rate of approximately:

> **77.5%**

compared with approximately:

> **78.0% on designated Validation**

The combined presentation therefore remained stable, although pure repeat-frequency ranking continued to produce higher next-basket accuracy.

## Final Recommendation Decision

### Repeat / Replenishment Channel

> **GO**

Customer Repeat Frequency is approved for operational next-product recommendation.

### Discovery / Cross-Sell Channel

> **CONDITIONAL GO**

Item-to-Item Affinity is approved for discovery recommendations, cross-sell experimentation and merchandising support.

It should not be interpreted as a guaranteed next-purchase predictor.

### Optional Unified List

The 70% Repeat / 30% Discovery blend may be used when a single recommendation list is required, but it remains a presentation layer rather than the core predictive architecture.

In [ ]:
# ================================================================
# 62.1 — FINAL RECOMMENDATION MODEL CARD
# ================================================================

final_recommendation_model_card = pd.DataFrame(
    {
        "Item": [
            "Business objective",
            "Prediction target",
            "Prediction horizon",
            "Repeat channel",
            "Discovery channel",
            "Optional combined presentation",
            "Repeat decision",
            "Discovery decision",
            "Repeat Holdout Hit Rate@10",
            "Repeat Holdout Hit Rate@20",
            "Repeat Eligible Hit Rate@20",
            "Discovery Holdout Hit Rate@10",
            "Discovery Holdout Hit Rate@20",
            "Hybrid Holdout Hit Rate@20",
            "Primary repeat limitation",
            "Primary discovery limitation",
            "Holdout status",
        ],

        "Value": [
            "Recommend products for the customer's next qualifying order",
            "Products appearing in next order",
            "Next order within 90 days",
            "Customer Repeat Frequency",
            "Item-to-Item Affinity",
            "70% Repeat / 30% Discovery",
            "GO",
            "CONDITIONAL GO",
            0.705912,
            0.792319,
            0.870896,
            0.216010,
            0.299238,
            0.774634,
            "Historical-frequency ranking may miss changing preferences",
            "Cross-sell affinity is less temporally stable and sparse-product performance is weaker",
            "CONSUMED",
        ],
    }
)


display(
    final_recommendation_model_card
)

In [ ]:
# ================================================================
# 62.2 — FINAL GOVERNANCE
# ================================================================

NOTEBOOK_6_COMPLETE = True

FINAL_REPEAT_DECISION = "GO"
FINAL_DISCOVERY_DECISION = "CONDITIONAL GO"


assert RECOMMENDATION_ARCHITECTURE_FROZEN is True
assert RECOMMENDATION_TEST_SET_OPENED is True
assert RECOMMENDATION_TEST_SET_CONSUMED is True


print(
    f"Repeat recommender: "
    f"{FINAL_REPEAT_DECISION}"
)

print(
    f"Discovery recommender: "
    f"{FINAL_DISCOVERY_DECISION}"
)

print(
    "✓ Recommendation holdout consumed."
)

print(
    "✓ No post-holdout tuning permitted."
)

print(
    "✓ Notebook 6 complete."
)

# Final Notebook Conclusion

## Next-Best-Product Recommendation

This notebook developed and validated a customer-level product recommendation architecture for predicting products likely to appear in the customer's next qualifying order.

The recommendation problem was separated into two distinct behaviours:

### Replenishment

Products previously purchased by the customer.

### Discovery

Products that are new to the customer.

The data supported this separation because next baskets contained substantial proportions of both repeat and new-to-customer products.

## Final Architecture

### Repeat Channel

> **Customer Repeat Frequency**

Products are ranked primarily by the number of historical customer orders containing the product, with recency used as a tie-breaker.

### Discovery Channel

> **Item-to-Item Affinity**

New products are ranked using cosine-style historical basket co-occurrence relationships between products already purchased by the customer and other catalogue products.

### Optional Combined Presentation

> **70% Repeat / 30% Discovery**

This may be used when the consuming application requires one combined list.

## Final Holdout Performance

The Repeat channel achieved approximately:

- **Top-5 Hit Rate: 59.6%**
- **Top-10 Hit Rate: 70.6%**
- **Top-20 Hit Rate: 79.2%**

Among episodes containing repeat products, Top-20 Hit Rate reached approximately:

> **87.1%**

The Discovery channel achieved approximately:

- **Top-5 Hit Rate: 15.4%**
- **Top-10 Hit Rate: 21.6%**
- **Top-20 Hit Rate: 29.9%**

The optional combined 70/30 list achieved approximately:

> **77.5% Hit Rate@20**

## Final Decision

### Repeat Recommendation

> **GO**

The replenishment channel demonstrated strong and highly stable out-of-time performance.

### Discovery Recommendation

> **CONDITIONAL GO**

The affinity discovery channel provides meaningful personalized cross-sell value but is less temporally stable than repeat recommendation.

It is best used for product discovery, cross-sell experimentation and merchandising support rather than guaranteed next-purchase prediction.

---

**Notebook 6 Status: COMPLETE**

In [ ]:
# ================================================================
# EXPORT AUDIT — NEXT BEST PRODUCT NOTEBOOK
# ================================================================

nbp_export_candidates = []


search_terms = [
    "recommend",
    "product",
    "rank",
    "score",
    "repeat",
    "discovery",
    "hybrid",
    "candidate",
]


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue


    columns = list(
        obj.columns
    )


    columns_lower = [
        str(column).lower()
        for column in columns
    ]


    has_customer = (
        "customer"
        in columns_lower
    )


    matching_columns = [
        column
        for column in columns
        if any(
            term
            in str(column).lower()
            for term in search_terms
        )
    ]


    if (
        has_customer
        and
        matching_columns
    ):

        nbp_export_candidates.append(
            {
                "DataFrame":
                    name,

                "Rows":
                    len(obj),

                "Matching Columns":
                    matching_columns,

                "All Columns":
                    columns,
            }
        )


nbp_export_candidates = pd.DataFrame(
    nbp_export_candidates
)


display(
    nbp_export_candidates
)

In [ ]:
# ================================================================
# EXPORT AUDIT 2 — HOLDOUT RECOMMENDATION PREDICTIONS
# ================================================================

print(
    "recommendation_holdout_predictions shape:",
    recommendation_holdout_predictions.shape
)

print(
    "\nColumns:"
)

for column in recommendation_holdout_predictions.columns:

    print(
        " -",
        column
    )


display(
    recommendation_holdout_predictions.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT AUDIT 3 — REPEAT CHANNEL HOLDOUT STRUCTURE
# ================================================================

print(
    "holdout_repeat_eligible shape:",
    holdout_repeat_eligible.shape
)

print(
    "\nColumns:"
)

for column in holdout_repeat_eligible.columns:

    print(
        " -",
        column
    )


display(
    holdout_repeat_eligible.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT AUDIT 4 — DISCOVERY CHANNEL HOLDOUT STRUCTURE
# ================================================================

print(
    "holdout_discovery_eligible shape:",
    holdout_discovery_eligible.shape
)

print(
    "\nColumns:"
)

for column in holdout_discovery_eligible.columns:

    print(
        " -",
        column
    )


display(
    holdout_discovery_eligible.head(
        10
    )
)

In [ ]:
# ================================================================
# EXPORT — NEXT BEST PRODUCT MODEL OUTPUT
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd

MODEL_OUTPUT_DIR = Path("model_outputs")

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Helper: normalise recommendation objects
# to clean Python lists of strings
# ------------------------------------------------

def normalise_product_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        (
            list,
            tuple,
            np.ndarray,
            pd.Series,
            set,
        )
    ):

        return [
            str(product)
            for product in value
            if pd.notna(product)
        ]

    if pd.isna(value):
        return []

    return [
        str(value)
    ]


# ------------------------------------------------
# Construct clean historical recommendation output
# ------------------------------------------------

next_best_product_scores = (
    recommendation_holdout_predictions[
        [
            "customer",
            "invoice",
            "order_date",
            "prediction_period",
            "region",
            "customer_order_number",
            "customer_total_orders",
            "history_depth_band",
            "historical_unique_product_count",
            "repeat_recommendations",
            "discovery_recommendations",
            "hybrid_recommendations",
        ]
    ]
    .copy()
)


# ------------------------------------------------
# Canonical Notebook 8 naming
# ------------------------------------------------

next_best_product_scores = (
    next_best_product_scores
    .rename(
        columns={
            "order_date":
                "prediction_date"
        }
    )
)


# ------------------------------------------------
# Standardise customer identifier
# ------------------------------------------------

next_best_product_scores[
    "customer"
] = (
    pd.to_numeric(
        next_best_product_scores[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


next_best_product_scores[
    "prediction_date"
] = pd.to_datetime(
    next_best_product_scores[
        "prediction_date"
    ]
)


# ------------------------------------------------
# Normalise recommendation lists
# ------------------------------------------------

for column in [
    "repeat_recommendations",
    "discovery_recommendations",
    "hybrid_recommendations",
]:

    next_best_product_scores[
        column
    ] = (
        next_best_product_scores[
            column
        ]
        .apply(
            normalise_product_list
        )
    )


# ------------------------------------------------
# Convenient Top-1 recommendations
# ------------------------------------------------

next_best_product_scores[
    "repeat_top1_product"
] = (
    next_best_product_scores[
        "repeat_recommendations"
    ]
    .apply(
        lambda products:
            (
                products[0]
                if len(products) > 0
                else pd.NA
            )
    )
)


next_best_product_scores[
    "discovery_top1_product"
] = (
    next_best_product_scores[
        "discovery_recommendations"
    ]
    .apply(
        lambda products:
            (
                products[0]
                if len(products) > 0
                else pd.NA
            )
    )
)


next_best_product_scores[
    "hybrid_top1_product"
] = (
    next_best_product_scores[
        "hybrid_recommendations"
    ]
    .apply(
        lambda products:
            (
                products[0]
                if len(products) > 0
                else pd.NA
            )
    )
)


# ------------------------------------------------
# Recommendation counts
# ------------------------------------------------

next_best_product_scores[
    "repeat_recommendation_count"
] = (
    next_best_product_scores[
        "repeat_recommendations"
    ]
    .apply(len)
)


next_best_product_scores[
    "discovery_recommendation_count"
] = (
    next_best_product_scores[
        "discovery_recommendations"
    ]
    .apply(len)
)


# ------------------------------------------------
# Explicit channel governance
# ------------------------------------------------

next_best_product_scores[
    "repeat_channel_status"
] = "GO"


next_best_product_scores[
    "discovery_channel_status"
] = "CONDITIONAL GO"


next_best_product_scores[
    "hybrid_channel_status"
] = (
    "OPTIONAL PRESENTATION LAYER"
)


# ------------------------------------------------
# Explicit model provenance
# ------------------------------------------------

next_best_product_scores[
    "recommendation_model_scope"
] = (
    "Products ranked immediately after a completed "
    "qualifying customer order for likely purchase "
    "within the following 90 days"
)


next_best_product_scores[
    "recommendation_score_provenance"
] = (
    "historical Jul-Aug performance holdout scoring"
)


next_best_product_scores[
    "recommendation_holdout_qualification"
] = (
    "Performance holdout; not fully pristine target-blind "
    "because descriptive holdout diagnostics were inspected "
    "before final evaluation"
)


# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert (
    next_best_product_scores[
        "customer"
    ]
    .notna()
    .all()
)


assert (
    next_best_product_scores[
        "prediction_date"
    ]
    .notna()
    .all()
)


assert (
    next_best_product_scores[
        [
            "customer",
            "invoice",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
), (
    "Expected one recommendation episode per customer/invoice."
)


assert (
    next_best_product_scores[
        "repeat_recommendation_count"
    ]
    .ge(0)
    .all()
)


assert (
    next_best_product_scores[
        "discovery_recommendation_count"
    ]
    .ge(0)
    .all()
)


# ------------------------------------------------
# Save persistent output
# ------------------------------------------------

nbp_output_path = (
    MODEL_OUTPUT_DIR
    /
    "next_best_product_scores.parquet"
)


next_best_product_scores.to_parquet(
    nbp_output_path,
    index=False
)


# ------------------------------------------------
# Export audit
# ------------------------------------------------

print(
    "✓ Next-Best-Product scores exported."
)

print(
    nbp_output_path.resolve()
)

print(
    f"Rows: "
    f"{len(next_best_product_scores):,}"
)

print(
    f"Customers: "
    f"{next_best_product_scores['customer'].nunique():,}"
)

print(
    f"Prediction date range: "
    f"{next_best_product_scores['prediction_date'].min()} "
    f"to "
    f"{next_best_product_scores['prediction_date'].max()}"
)

print(
    f"Repeat Top-1 coverage: "
    f"{next_best_product_scores['repeat_top1_product'].notna().mean():.2%}"
)

print(
    f"Discovery Top-1 coverage: "
    f"{next_best_product_scores['discovery_top1_product'].notna().mean():.2%}"
)


display(
    next_best_product_scores.head(
        10
    )
)